# Notes

In [ ]:
# Lower body
    # temporal 
        # every temporal parameter in seconds (later, apart from stride time every other thing should also be calcualted as % of stride time)
        # round temporal parameters to 3 digits
        # double support time has to be rewieved (2 stages for each cycle)

    # spatial
        # every spacial parameter in meters (round to 3 digits)
        # + foot progression angle [deg] (toe in / toe out in gaitrite, only contralateral can be calculated for each gait cycle)
            # at the end it has to be per side
        # foot height during swing (no precise def yet)
        
        # Rotations
        # possibly in 3D, joint/pateient centered coord system
        # ankle angle [deg]
        # knee angle [deg]
        # hip [deg] 

# Upper body
    # spatial
        # trunk ROM 3D [deg] (torsion included)
        # head ROM 3D [deg] (torsion included)
        # arm ROM [deg] in 2 planes (plane of progression, vertical plane tbd)
        # wrist angle [deg] 3D


# a class for every gait parameter
   

# first publication 
    # knee rom 
    # arm swing
    # kinect vs stereo 
    # stat: ICC3 in pingiun table (3rd options)

# Imports

In [2]:
## conda env: stereo_vision
import numpy as np
from rtmlib import draw_skeleton, draw_bbox
from Util.util import *
from Util.calibration_util import *
from Util.gait_util import *
from Util.statistics_util import *
import glob
import time

from IPython.display import display, clear_output
from scipy.signal import resample

import pyzed.sl as sl
import matplotlib.pyplot as plt # TODO: get the 5.11 SDK working

# load model info (config, input size, etc)
with open('./Util/2d_model/models.json') as f:
    models = json.load(f)

# load static properties for gait analysis
with open("./Util/gait_analysis_properties.json", "r") as json_file:
    gait_analysis_properties = json.load(json_file)
    stereo_properties = gait_analysis_properties["stereo"]
    qualisys_properties = gait_analysis_properties["qualisys"]

%matplotlib ipympl

# get ZED SDK verison
print(f"ZED SDK: {sl.Camera.get_sdk_version()}")

ZED SDK: 5.0.3


In [3]:
participant = "mate_unger"
# participant = "max_wuehr"
paradigm = "normal_walking"

In [ ]:
# from export_stereo_video import *

export_svo_avi(
    input_path="D:\\mwuehr\\stereolabs_valdiation\\data\\vasil_derventliev\\stereolabs\\E4F91ED1-FD25-4A0C-BD82-11BC643510E8\\43916681.svo2",
    output_path="D:\\mwuehr\\stereolabs_valdiation\\data\\vasil_derventliev\\stereolabs\\E4F91ED1-FD25-4A0C-BD82-11BC643510E8\\43916681.avi"
)

# 2D Keypoints

## Model setup

In [ ]:
device = "cuda"  # cpu, cuda, mps
backend = "onnxruntime"  # opencv, onnxruntime, openvino
detector_name = 'YOLOX_nano' # 'YOLOX_l_COCO','YOLOX_nano','YOLOX_tiny','YOLOX_s','YOLOX_m','YOLOX_l','YOLOX_x'
pose_name = 'RTMPose_x' # (26) 'RTMPose_t', 'RTMPose_s', 'RTMPose_m', 'RTMPose_l', 'RTMPose_m2', 'RTMPose_l2', 'RTMPose_x', (133) 'RTMW_l', 'RTMW_x'
labels = models['pose_models']["26"]["kpt_labels"]

custom = Custom(det_class='YOLOX', #'RTMDet',
                det=models['detectors'][detector_name]['path'],
                det_input_size=models['detectors'][detector_name]['input_size'],
                pose_class='RTMPose',
                pose=models['pose_models']["26"]["models"][pose_name]['path'],
                pose_input_size=models['pose_models']["26"]["models"][pose_name]['input_size'],
                backend=backend,
                device=device) 

#TODO: fix CUDA not being used

## Flicker filtering

### Image normailazion

In [ ]:
no_ir_path = f"no_ir.png"
ir_path = f"ir.png"

no_ir_img = cv2.imread(no_ir_path,1)
frame = cv2.imread(ir_path,1)
h,w = no_ir_img.shape[:2]

r_mean, r_std = cv2.meanStdDev(frame[:,:,0])
r_ch = (frame[:,:,0] - r_mean) / r_std

g_mean, g_std = cv2.meanStdDev(frame[:,:,1])
g_ch = (frame[:,:,1] - g_mean) / g_std

b_mean, b_std = cv2.meanStdDev(frame[:,:,2])
b_ch = (frame[:,:,2] - b_mean) / b_std

frame = np.dstack((r_ch,g_ch,b_ch))

# print(cv2.meanStdDev(no_ir_img))
# print(cv2.meanStdDev(ir_img))
# print("")
# print(cv2.meanStdDev(no_ir_eq))
# print(cv2.meanStdDev(ir_eq))


# no_ir_res = np.hstack((no_ir_img, no_ir_eq))
# ir_res = np.hstack((ir_img, ir_eq))

# cv2.imwrite("no_ir_eq.png", no_ir_res)
# cv2.imwrite("ir_eq.png", ir_res)

cv2.imshow("image", cv2.resize(frame, (int(w / 2), int(h / 2))))
cv2.waitKey(0)
cv2.destroyAllWindows()

In [ ]:
frame = cv2.imread("./ir.png", 1)
no_ir_img = cv2.imread("./no_ir.png", 1)

# h,w = img.shape[:2]

r = cv2.equalizeHist(frame[:,:,0])
g = cv2.equalizeHist(frame[:,:,1])
b = cv2.equalizeHist(frame[:,:,2])
ir_eq = np.dstack((r,g,b))

r = cv2.equalizeHist(no_ir_img[:,:,0])
g = cv2.equalizeHist(no_ir_img[:,:,1])
b = cv2.equalizeHist(no_ir_img[:,:,2])
no_ir_eq = np.dstack((r,g,b))


ir_res = np.hstack((frame, ir_eq))
no_ir_res = np.hstack((no_ir_img, no_ir_eq))

print(cv2.meanStdDev(no_ir_img))
print(cv2.meanStdDev(frame))
print("")
print(cv2.meanStdDev(no_ir_eq))
print(cv2.meanStdDev(ir_eq))

cv2.imwrite("no_ir_ch_eq.png", no_ir_res)
cv2.imwrite("ir_ch_eq.png", ir_res)

# cv2.imshow("image", cv2.resize(img_eq, (int(w / 2), int(h / 2))))
# cv2.waitKey(0)
# cv2.destroyAllWindows()

### Image histograms

In [ ]:
import matplotlib.pyplot as plt
import cv2

no_ir = cv2.imread('no_ir.png')
ir = cv2.imread('ir.png')

# calculate mean value from RGB channels and flatten to 1D array
# no_ir_values = no_ir_img.mean(axis=2).flatten()
# ir_values = ir_img.mean(axis=2).flatten()

r_ir = ir[:,:,0]
g_ir = ir[:,:,1]
b_ir = ir[:,:,2]

r_no_ir = no_ir[:,:,0]
g_no_ir = no_ir[:,:,1]
b_no_ir = no_ir[:,:,2]


r_ir_count, r_ir_bins = np.histogram(a=r_ir,bins=25)
r_no_ir_count, r_no_ir_bins = np.histogram(a=r_no_ir,bins=25)

g_ir_count, g_ir_bins = np.histogram(a=g_ir,bins=25)
g_no_ir_count, g_no_ir_bins = np.histogram(a=g_no_ir,bins=25)

b_ir_count, b_ir_bins = np.histogram(a=b_ir,bins=25)
b_no_ir_count, b_no_ir_bins = np.histogram(a=b_no_ir,bins=25)


# plt.close('all')
# fig, axs = plt.subplots(1,3)
# axs[0].bar(r_ir_bins, r_ir_count, alpha=0.2)

# plt.show()
# plt.tight_layout()
r_ir_count, r_ir_bins

### Denoising

In [ ]:
import matplotlib.pyplot as plt
import cv2

no_ir_img = cv2.imread('no_ir.png')
frame = cv2.imread('ir.png')

# dst = cv.fastNlMeansDenoisingColored(img,None,10,10,7,21)
dst = cv2.fastNlMeansDenoisingColored(frame,None,10,10,7,7)
res = np.hstack((no_ir_img, dst))
h,w = res.shape[:2]
cv2.imshow("image",cv2.resize(res, (int(w / 3), int(h / 3))))
cv2.waitKey(0)
cv2.destroyAllWindows()

### Windowing

In [ ]:
cap = cv2.VideoCapture(f".\\stereo_videos\\validation_test\\{participant}\\{paradigm}\\43916681.avi")
if not cap.isOpened():
    print("Error opening video file")

means_and_stds = []
i = 0
# Read until video is completed
while i<int(cap.get(cv2.CAP_PROP_FRAME_COUNT)):

    # Capture frame-by-frame
    ret, frame = cap.read()

    if ret == True and frame is not None:
        h,w = frame.shape[:2]
        i+=1

        # cv2 has BGR as default
        b_mean, b_std = cv2.meanStdDev(frame[:,:,0])
        g_mean, g_std = cv2.meanStdDev(frame[:,:,1])
        r_mean, r_std = cv2.meanStdDev(frame[:,:,2])
        
        means_and_stds.append([r_mean, g_mean, b_mean, r_std, g_std, b_std])

means_and_stds = np.squeeze(np.asarray(means_and_stds))

In [ ]:
windowed = []
window_mean = []
diffs = []

window_len = 60

pad_widths = ((int(window_len/2),int(window_len/2)),(0,0))
padded_means_and_stds = np.pad(array=means_and_stds,pad_width=pad_widths, mode='symmetric')

for frame_idx in range(len(padded_means_and_stds)-window_len):
    windowed_mean = np.mean(padded_means_and_stds[frame_idx:frame_idx + window_len,:3])
    windowed_std = np.mean(padded_means_and_stds[frame_idx:frame_idx + window_len,3:])

    mean = np.mean(padded_means_and_stds[frame_idx,:3])
    diff = windowed_mean - mean

    windowed.append(mean+diff)
    window_mean.append(mean)
    diffs.append(diff)

windowed = np.asarray(windowed)
window_mean = np.asarray(window_mean)
diffs = np.asarray(diffs)
diffs = np.round(diffs).astype(int)

In [ ]:
ch_avg = np.mean(padded_means_and_stds[:,:3], axis=1)

fig, axs = plt.subplots(3,1,figsize=(15,6))
axs[0].plot(padded_means_and_stds[:,0],c='#f00000', alpha=0.6)
axs[0].plot(padded_means_and_stds[:,1],c='#43d12a', alpha=0.6)
axs[0].plot(padded_means_and_stds[:,2],c='#0800f0', alpha=0.6)
axs[0].plot(ch_avg,'k')

axs[1].plot(padded_means_and_stds[:,3],c='#f00000', alpha=0.6)
axs[1].plot(padded_means_and_stds[:,4],c='#43d12a', alpha=0.6)
axs[1].plot(padded_means_and_stds[:,5],c='#0800f0', alpha=0.6)

axs[2].plot(windowed[:])
# axs[2].plot(window_mean, alpha=0.4)
axs[0].set_ylim([75,105])
# axs[2].set_ylim([75,105])

In [ ]:
cap = cv2.VideoCapture(f".\\stereo_videos\\validation_test\\{participant}\\{paradigm}\\43916681.avi")
if not cap.isOpened():
    print("Error opening video file")

means_and_stds = []
num_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(num_frames)
i = 0
# Read until video is completed
while i<num_frames:
    # Capture frame-by-frame
    ret, frame = cap.read()

    if ret == True and frame is not None:
        h,w = frame.shape[:2]
        i+=1

        frame_show = frame - (np.ones_like(frame)*diffs[i]).astype(np.uint8)
        cv2.imshow("Image", cv2.resize(frame_show, (int(w / 2), int(h / 2))))

                # cv2 has BGR as default
        b_mean, b_std = cv2.meanStdDev(frame[:,:,0])
        g_mean, g_std = cv2.meanStdDev(frame[:,:,1])
        r_mean, r_std = cv2.meanStdDev(frame[:,:,2])
        
        means_and_stds.append([r_mean, g_mean, b_mean, r_std, g_std, b_std])


        # Press Q on keyboard to exit
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    else:
        break

cap.release()
cv2.destroyAllWindows()

means_and_stds = np.squeeze(np.asarray(means_and_stds))


In [ ]:
fig, axs = plt.subplots(2,1,figsize=(15,6))
axs[0].plot(means_and_stds[:,0],c='#f00000', alpha=0.6)
axs[0].plot(means_and_stds[:,1],c='#43d12a', alpha=0.6)
axs[0].plot(means_and_stds[:,2],c='#0800f0', alpha=0.6)

axs[1].plot(means_and_stds[:,3],c='#f00000', alpha=0.6)
axs[1].plot(means_and_stds[:,4],c='#43d12a', alpha=0.6)
axs[1].plot(means_and_stds[:,5],c='#0800f0', alpha=0.6)

## 2D inference

In [ ]:
input_folder = f".\\stereo_videos\\validation_test\\{participant}\\{paradigm}"



for g in glob.glob(os.path.join(input_folder, "*.avi")):
    print(f"processing: {g}")

    keypoints_2d = []
    confidence_2d = []
    color_2d = []
    
    cap = cv2.VideoCapture(g)
    if cap.isOpened() == False:
        print("Error opening video file")

    total_num_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    processed_num_frames = 0
    start_time = time.time_ns()

    # Read until video is completed
    while cap.isOpened():

        # Capture frame-by-frame
        ret, frame = cap.read()

        if ret == True and frame is not None:
            processed_num_frames += 1
            width = frame.shape[1]
            height = frame.shape[0]

            # frame = cv2.fastNlMeansDenoisingColored(frame,None,10,10,7,7)


            # inference
            keypoints_2d, scores = custom(frame)

            # visualize
            boxes = [pose_to_bbox(x) for x in keypoints_2d]
            img_show = draw_bbox(frame, boxes, (0, 0, 255))
            # img_show = draw_skeleton(img_show, keypoints, kpt_labels)
            for kpt in keypoints_2d[0,:,:]:
                cv2.circle(img_show, (int(kpt[0]), int(kpt[1])), 5, (0, 255, 0), 3)

            cv2.imshow("Image", cv2.resize(img_show, (int(width / 2), int(height / 2))))

            ## store results (frames * kpt * (x,y))
            # remove multiperson cases
            if keypoints_2d.shape != (1, 26, 2):
                keypoints_2d = keypoints_2d[0, :, :]
                scores = scores[0, :]

            keypoints_2d = np.squeeze(keypoints_2d)
            color = np.array([frame[(int(x),int(y))] for x,y in keypoints_2d[:,:2]])
            
            keypoints_2d.append(keypoints_2d)
            confidence_2d.append(np.squeeze(scores))
            color_2d.append(color)

            # display how much time elapsed/is left
            clear_output(wait=True)
            end_time = time.time_ns()
            elapsed_time = (end_time - start_time) / 1000000000

            if int(elapsed_time % 5) == 0:

                fps = round(processed_num_frames / elapsed_time, 2)
                done_ratio = processed_num_frames / total_num_frames
                expected_duration_min = elapsed_time / (done_ratio * 60)

                display(
                    f"exp_duration: {round(expected_duration_min,2)} minutes, elapsed: {round(elapsed_time/60,2)} minutes"
                )
                display(f"done: {round(done_ratio*100,2)}%, fps: {fps}")

            # Press Q on keyboard to exit
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break

        else:
            break

    cap.release()
    cv2.destroyAllWindows()

    keypoints_2d = np.asarray(keypoints_2d)
    confidence_2d = np.asarray(confidence_2d)
    color_2d = np.asarray(color_2d)

    print(f"shape of keypoints_2d:  {keypoints_2d.shape} (frames * kpt * (x, y))")
    print(f"shape of confidence_2d: {confidence_2d.shape} (frames * kpt)")
    print(f"shape of color_2d:      {color_2d.shape} (frames * kpt * rgb)")

    

## Filter data 
(so it's less jiggly for depth)

In [ ]:
# see how many ppl have been detected, should be just 1 
for idx, kpt in enumerate(keypoints_2d):
    print(idx, kpt.shape)
    break

In [ ]:
filtered_2d_keypoints = []

for keypoint_id, keypoint in enumerate(keypoints_2d.transpose(1,0,2)):
    result = filter_2d_keypoint(keypoint,6,7, 60, False,f'{keypoint_id}')
    filtered_2d_keypoints.append(result)

filtered_2d_keypoints = np.array(filtered_2d_keypoints).transpose(1,0,2)
print("Filtered data shape:", filtered_2d_keypoints.shape)

## Save data

In [ ]:
out_file_name = os.path.join(input_folder, f"{os.path.basename(g).split('.')[0]}_test.npz")

with open(out_file_name, "w") as f:
    np.savez(
        out_file_name,
        keypoints_2d=keypoints_2d,
        keypoints_2d_filtered=filtered_2d_keypoints, 
        # confidence_2d=confidence_2d,
        # color=color_2d,
        labels=labels
    )

print(f"Saved 2d keypoint to {out_file_name}")

## Visualization

In [ ]:
input_vid = f".\\stereo_videos\\validation_test\\{participant}\\{paradigm}\\43916681.avi"

vid_basename = os.path.basename(input_vid)
folder_name = os.path.dirname(input_vid)
npz_basename = f"{vid_basename.split(".")[0]}.npz"

load_path = os.path.join(folder_name, npz_basename)
loaded_data = np.load(load_path)
print(f"loaded data : {load_path}\nwith keys: {list(loaded_data.keys())}")

keypoints_2d = loaded_data["keypoints_2d"]
kpt_labels = loaded_data["kpt_labels"]



cap = cv2.VideoCapture(input_vid)
if cap.isOpened() == False:
    print("Error opening video file")

total_num_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
frame_idx = 0

# Read until video is completed
while cap.isOpened() and frame_idx < keypoints_2d.shape[0]-1:

    # Capture frame-by-frame
    ret, frame = cap.read()

    if ret == True and frame is not None:
        frame_idx += 1
        width = frame.shape[1]
        height = frame.shape[0]


        for kpt in keypoints_2d[frame_idx,:,:]:
            cv2.circle(frame,(int(kpt[0]),int(kpt[1])),radius=3, color=(0,0,255), thickness=2 )
            # cv2.putText(frame, f"{conf:.2f}",(int(kpt[0]),int(kpt[1])),cv2.FONT_HERSHEY_SIMPLEX,fontScale=0.75,color = (255, 255, 255))
            

        cv2.imshow("Image", cv2.resize(frame, (int(width / 2), int(height / 2))))
        time.sleep(0.01)

        # Press Q on keyboard to exit
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    else:
        break

cap.release()
cv2.destroyAllWindows()
    

# 3D Keypoints

## Load 2D data

In [ ]:
npz_load_path = f"stereo_videos\\validation_test\\{participant}\\{paradigm}\\43916681_test.npz"
loaded_data = np.load(npz_load_path)

svo_path = f"{os.path.splitext(npz_load_path)[0]}.svo2"
if not os.path.exists(svo_path):
    print(f".svo2 file is missing: \n{svo_path}\n")
keypoints_2d = loaded_data["keypoints_2d"]
keypoints_2d = loaded_data['keypoints_2d_filtered']
# confidence_2d = loaded_data["confidence_2d"]
# color = loaded_data["color"]
labels = loaded_data["labels"]

print(f"loaded data : {npz_load_path}\nwith keys: {list(loaded_data.keys())}")
print(f"2d keypoint shape: {keypoints_2d.shape}")


## Get depth from StereoLabs 

In [ ]:
# TODO: update ZED sdk, try it with latest models, methods

keypoints_3d = []
keypoints_point_cloud = []

# Create a ZED camera object
zed = sl.Camera()

input_type = sl.InputType()
input_type.set_from_svo_file(svo_path)  # Set init parameter to run from the .svo
init_parameters = sl.InitParameters(input_t=input_type, svo_real_time_mode=False)

init_parameters.depth_mode = sl.DEPTH_MODE.NEURAL_PLUS
init_parameters.coordinate_units = sl.UNIT.METER  # CENTIMETER, METER, MILLIMETER
# init_parameters.coordinate_system = sl.COORDINATE_SYSTEM.RIGHT_HANDED_Y_UP

# TODO: see how own filtering does without this built in one
init_parameters.depth_stabilization = 75 # 0 to trun it off, otherwise 1-100 linear. default is 30

# Open the ZED
err = zed.open(init_parameters)
framerate = zed.get_camera_information().camera_configuration.fps
resolution = zed.get_camera_information().camera_configuration.resolution

svo_depth = sl.Mat()
svo_image = sl.Mat()
# svo_confidance = sl.Mat()
# svo_xyz = sl.Mat()

frame_idx = 0

while frame_idx < keypoints_2d.shape[0] - 1 and zed.grab() == sl.ERROR_CODE.SUCCESS:

    current_keypoints = []
    current_xyz = []

    # Get frame count
    frame_idx = zed.get_svo_position()

    # get color image and depth map
    zed.retrieve_measure(svo_depth, sl.MEASURE.DEPTH)
    zed.retrieve_image(svo_image, sl.VIEW.LEFT) # BGRA image
    # zed.retrieve_measure(svo_confidance, sl.MEASURE.CONFIDENCE)
    # zed.retrieve_measure(svo_xyz, sl.MEASURE.XYZRGBA) # trash documentation, no clue what this returns

    img = svo_image.get_data()
    depth_map = svo_depth.get_data()
    depth_map = np.transpose(depth_map)
    # depth_confidance_map = svo_confidance.get_data()
    # point_cloud = svo_xyz.get_data()
    # print(f"point_cloud shape: {point_cloud.shape}")
    

    # get coords for all keypoints within the current frame
    # note: x,y coords are in pixels, depth is in meters
    for x, y in keypoints_2d[frame_idx, :]:

        depth = depth_map[int(x), int(y)]
        # b, g, r, a = img[int(x), int(y)]
        # depth_conf = depth_confidance_map[int(x), int(y)]
        # xyz_point = point_cloud[int(x), int(y)]

        cv2.circle(img, (int(x), int(y)), 5, (0, 0, 100 * 255 / 100), 3)

        current_keypoints.append(np.array([x, y, depth]))
        # current_xyz.append(xyz_point)
        

        
        # print(f"x: {x:.0f}, y: {y:.0f}, depth: {depth:.01f}, xy_conf: {xy_conf:.01f}, depth_conf: {depth_conf:.0f}")

    current_keypoints = np.array(current_keypoints)
    keypoints_3d.append(current_keypoints)
    # keypoints_point_cloud.append(current_xyz)

#     ## show frame
    cv2.imshow("vid", cv2.resize(img, (800, 600)))

    if cv2.waitKey(25) & 0xFF == ord("q"):
        break


cv2.destroyAllWindows()
zed.close()

# shape: frames * kpts * (x, y, depth,xy_conf, depth_conf)
keypoints_3d = np.array(keypoints_3d)
print(f"keypoints 3d shape: {keypoints_3d.shape}")

# keypoints_point_cloud = np.array(keypoints_point_cloud)
# print(f"keypoints_xyz shape: {keypoints_xyz.shape}")

## Get camera position (R,t)

In [ ]:
from Util.calibration_util import Pattern, get_image_points
import glob

# input/output folders
data_dir = f"./stereo_videos\\validation_test\\{participant}\\{paradigm}"
output_dir = data_dir

with open("./Util/camera_params.json", "r") as f:
    intrinsic_params = json.load(f)

# describe real world calibration pattern parameteres
board_pattern = Pattern(6, 4, 0.150)


# serial numbers, file names and images
sns_fns_imgs = [
    (
        f"SN43916681",
        file.split(".png")[0].split("\\")[1],
        cv2.imread(file, cv2.IMREAD_GRAYSCALE),
    )
    for file in glob.glob(os.path.join(data_dir, "*.png"))
]

extrinsics = []
camera_coords = []


for sn, fn, img in sns_fns_imgs:
    print(f"Processing {fn}...")
    img_points, obj_points = get_image_points(img, fn, output_dir, board_pattern, save_output=False)
    print(f"sn: {sn}")
    cameraMatrix = np.array(intrinsic_params[sn]["left_sensor"]["camera_matrix"])
    distCoeffs = np.array(intrinsic_params[sn]["left_sensor"]["distortion_coeff"])

    f_x = cameraMatrix[0, 0]
    f_y = cameraMatrix[1, 1]
    c_x = cameraMatrix[0, 2]
    c_y = cameraMatrix[1, 2]

    success, rvec, t = cv2.solvePnP(
        obj_points,
        img_points,
        cameraMatrix,
        distCoeffs,
        useExtrinsicGuess=False,
        flags=cv2.SOLVEPNP_ITERATIVE,
    )  # CV_P3P ,CV_EPNP
    # success, rvec, tvec, inliners = cv.solvePnPRansac(obj_points, img_points,cameraMatrix,distCoeffs)
    if success:
        print(f"{fn}... OK")
        R, jac = cv2.Rodrigues(rvec)

        Xw = -np.matrix(R).T * np.matrix(t)
        camera_coords.append(Xw)

        print(f"distance: {round(np.linalg.norm(Xw),2)} m")

    else:
        print(f"{fn}... FAILED")


## Transform 3D keypoints to world-coordinate system

In [ ]:
keypoints_3d_world_frame = []

for kpt in keypoints_3d.transpose(1,2,0):

    u = kpt[0,:]
    v = kpt[1,:]
    Zc = kpt[2,:]

    Xw = ((u-c_x) * Zc/f_x) - t[0]
    Yw = ((v-c_y) * Zc/f_y) - t[1]
    Zw = Zc - t[2]

    # multiply by -1 so axes go in the correct direction
    Yw = -Yw
    Zw = -Zw
    
    keypoints_3d_world_frame.append(np.array([Zw,Xw,Yw]))

    
keypoints_3d_world_frame = np.array(keypoints_3d_world_frame)
keypoints_3d_world_frame.shape 

## Save data

In [ ]:
save_path = f"stereo_videos\\validation_test\\{participant}\\{paradigm}\\43916681_test.npz"

with open(save_path, "w") as f:
    np.savez(
        save_path,
        keypoints_2d=keypoints_2d,
        keypoints_2d_filtered=keypoints_2d,
        keypoints_3d=keypoints_3d,
        # confidence_2d=confidence_2d,
        labels=labels,
        # keypoints_3d=keypoints_3d_world_frame,
        # R = R,
        # t = t
    )

print(f"Saved 3D keypoints to {save_path}")

In [ ]:
npz_load_path = f"stereo_videos\\validation_test\\{participant}\\{paradigm}\\43916681_test.npz"
loaded_data = np.load(npz_load_path)

keypoints_2d = loaded_data["keypoints_2d"]
keypoints_2d = loaded_data['keypoints_2d_filtered']
keypoints_3d = loaded_data['keypoints_3d']
labels = loaded_data["labels"]

print(f"loaded data : {npz_load_path}\nwith keys: {list(loaded_data.keys())}")

## Filtering +  Qualisys - Stereo comparison

In [ ]:
# find the only file in the participant's folder with the .tsv extension
qualisys_tsv_filename = [g for g in glob.glob(os.path.join("stereo_videos","validation_test",participant,paradigm,"*.tsv"))][0]
print(f"converting: {qualisys_tsv_filename}")

# convert .tsv qualisys export to npz file
format_qualisys_export(qualisys_tsv_filename)

In [ ]:
# load QUALISYS data
file_path = f"stereo_videos\\validation_test\\{participant}\\{paradigm}\\qualisys.npz"
data = np.load(file_path, allow_pickle=True)
# qualisys_data = data['pose_data']
qualisys_data = data["keypoints"]
qualisys_labels = data['labels']
print(f"Qualisys data keys: {list(data.keys())}")
print(f"Qualisys data shape: {qualisys_data.shape}")

In [ ]:
## load STEREO data
npz_load_path = f"stereo_videos\\validation_test\\{participant}\\{paradigm}\\43916681_test.npz"
loaded_data = np.load(npz_load_path)

keypoints_2d = loaded_data["keypoints_2d"]
# confidence_2d = loaded_data["confidence_2d"]
keypoints_3d = loaded_data["keypoints_3d"]
stereo_labels = list(loaded_data["labels"])
# R = loaded_data["R"]
# t = loaded_data["t"]

print(f"loaded data : {npz_load_path}\nwith keys and shapes:")
[f"{key}:    {loaded_data[key].shape}" for key in list(loaded_data.keys())]



In [ ]:
# stereo labels
for idx,label in enumerate(kpt_labels):
    print(idx, label)

In [ ]:
# Qualisys labels
for i,l in enumerate(data['kpt_labels']):
    print(i,l)

In [ ]:
plt.figure(figsize=(15,2))
plt.plot(Sx)
# plt.plot(Qx)
plt.show()

In [ ]:
# perform filtering, visualize results as a comparison between Qualisys and Stereo
from scipy.signal import butter, filtfilt

# keypoints_3d = loaded_data["keypoints_3d"]



# keypoints_3d[:,:,890:935] = np.nan
# keypoints_3d[:,:,1235:1280] = np.nan
# keypoints_3d[:,:,1585:1630] = np.nan

keypoint_name = "right_ankle"
contra_keypoint_name = "left_ankle"

# Qualisys
qualisys_idx = list(qualisys_labels).index(keypoint_name)
qualisys_fps = 100
qualisys_data = filter_data(qualisys_data, qualisys_fps, "lowpass", [7], 4, 0.25)

Qd = qualisys_data[qualisys_idx, 0, :]
Qx = qualisys_data[qualisys_idx, 2, :]
Qy = qualisys_data[qualisys_idx, 1, :]


# stereo
stereo_idx = stereo_labels.index(keypoint_name)
stereo_fps = 100
# keypoints_3d_filtered = filter_data(keypoints_3d, stereo_fps, "lowpass", [2], 4, 0.25)
keypoints_3d_filtered = keypoints_3d_100.transpose(1,2,0)
# keypoints_3d_filtered = filter_data(keypoints_3d, stereo_fps, "lowpass", [7], 4, 0.25)

#TODO: try to figure out why bandpass makes it broken
# keypoints_3d_filtered[:, :2, :] = filter_data(copy.deepcopy(keypoints_3d_filtered), 60, "lowpass", [2], 4, 0.25)[:, :2, :]
# keypoints_3d_filtered[:, 2, :] = filter_data(copy.deepcopy(keypoints_3d_filtered), 60, "bandpass", [0.2, 2], 4, 0.25)[:, 2, :]


shift = 200
Sd = keypoints_3d_filtered[stereo_idx, 0, shift:]
Sx = keypoints_3d_filtered[stereo_idx, 1, shift:]
Sy = keypoints_3d_filtered[stereo_idx, 2, shift:]


## plot
tQ = np.linspace(0, len(Qd) / qualisys_fps, len(Qd))
tS = np.linspace(0, len(Sx) / stereo_fps, len(Sx))
xticks = np.array([0, 11, 16, 18, 23, 25, 29, 31, 36, 38, 43, 45, 50, 52, 57, 59, 64, 66])  # manual annotation
plt.close("all")
fig, axs = plt.subplots(6, 1, figsize=(13, 8))


## Horizontal ax 1-------------------------------------------------------
axs[0].plot(tQ, Qy, "m", label="Gold")
# axs[0].plot(tQ[1:],np.diff(Qy),'m',label = 'Gold')
axs[1].plot(tS, Sx, "m", label="Stereo")
# axs[1].plot(tS[1:],np.diff(Sx),'m',label='Stereo')
# axs[1].plot(tS,keypoints_3d[stereo_idx,1,shift:],"g--",alpha=0.5,label="raw")

## Horizontal ax 2-------------------------------------------------------
# axs[2].plot(tQ, Qd, label="Gold")
axs[2].plot(tQ[1:],np.diff(Qd),label='Gold')
# axs[3].plot(tS, Sd, label="Stereo")
axs[3].plot(tS[1:],np.diff(Sd),label='Stereo')
# axs[3].plot(tS,keypoints_3d[stereo_idx,0,shift:],"g--",alpha=0.5,label="raw")

## Vertical ax ----------------------------------------------------------
# axs[4].plot(tQ, Qx, "r", label="Gold")
axs[4].plot(tQ[1:],np.diff(Qx),'r',label = 'Gold')
# axs[5].plot(tS, Sy, "r", label="Stereo")
axs[5].plot(tS[1:],np.diff(Sy),'r',label='Stereo')
# axs[5].plot(tS,keypoints_3d[stereo_idx,2,shift:],"g--",alpha=0.5,label="raw")


plt.suptitle(keypoint_name)
axs[0].set_title("X axis (horizontal)")
axs[2].set_title("Y axis (horizontal)")
axs[4].set_title("Z axis (vertical)")
for ax in axs:
    ax.legend(loc="upper left")
    ax.set_xlim([min(xticks), max(xticks)])
    ax.set_xticks(xticks - 3.3, xticks)
    ax.grid()

plt.tight_layout()

In [ ]:
# save filtered stereo data
stereo_save_path = f"stereo_videos\\validation_test\\{participant}\\{paradigm}\\43916681.npz"
with open(stereo_save_path, "w") as f:
    np.savez(
        stereo_save_path,
        keypoints_2d=keypoints_2d,
        confidence_2d=confidence_2d,
        kpt_labels=kpt_labels,
        keypoints_3d=keypoints_3d,
        keypoints_3d_filtered=keypoints_3d_filtered,
        R = R,
        t = t
    )

print(f"Saved stereo 3D keypoints to: {stereo_save_path}")

# save filtered qualisys data
qualisys_save_path = f"stereo_videos\\validation_test\\{participant}\\{paradigm}\\qualisys.npz"
with open(qualisys_save_path, "w") as f:
    np.savez(
        qualisys_save_path,
        keypoints=qualisys_data,
        kpt_labels=qualisys_labels,
    )

print(f"Saved qualisys 3D keypoints to: {qualisys_save_path}")

# Gait analysis

## Load data

In [ ]:
## load STEREO data
stereo_file_path = f"stereo_videos\\validation_test\\{participant}\\{paradigm}\\43916681.npz"
loaded_stereo_data = np.load(stereo_file_path)

stereo_data = loaded_stereo_data["keypoints_3d_filtered"]
stereo_labels = list(loaded_stereo_data["kpt_labels"])

print(f"stereo data: {stereo_data.shape}")
print(f"stereo_labels: {len(stereo_labels)}")
for prop, val in stereo_properties.items():
    print(f"{prop:<13} {val:.2f}")

print("\n \n")

# load QUALISYS data
qualisys_file_path = f"stereo_videos\\validation_test\\{participant}\\{paradigm}\\qualisys.npz"
loaded_qualisys_data = np.load(qualisys_file_path, allow_pickle=True)
# convert values to meters
qualisys_data = loaded_qualisys_data["keypoints"]
qualisys_labels = loaded_qualisys_data["kpt_labels"].tolist()


print(f"qualisys_data: {qualisys_data.shape}")
print(f"qualisys_labels: {len(qualisys_labels)}")
for prop, val in qualisys_properties.items():
    print(f"{prop:<13} {val:.2f}")

## Get gait events, analyze result

### Qualisys

In [ ]:
qualisys_data[:,1,:] = qualisys_data[:,1,:] * (-1)
# qualisys_data[:,0,:] = qualisys_data[:,0,:] * (-1)
# qualisys_data[:,2,:] = qualisys_data[:,2,:] * (-1)

In [ ]:
# define where results (figures, csv tables) are saved
results_folder = os.path.join("stereo_videos", "validation_test",participant, paradigm,"results")
if not os.path.exists(results_folder):
    os.mkdir(results_folder)

qualisys_events = get_gait_events(
    keypoint_data=qualisys_data,
    kpt_labels=qualisys_labels,
    gait_analysis_properties=qualisys_properties,
    debug_figs_file_path=os.path.join(results_folder,"qualisys.npz"),
)
qualisys_results = gait_analysis(
    keypoint_data=qualisys_data,
    gait_events=qualisys_events,
    kpt_labels=qualisys_labels,
    gait_analysis_properties=qualisys_properties,
    debug_file_path=os.path.join(results_folder,"qualisys.npz")
)
qualisys_result_table, qualisys_result_df = display_results(qualisys_results)

### Stereo

In [ ]:
stereo_data[:,1,:] = stereo_data[:,1,:] * (-1)
stereo_data.shape

In [ ]:
stereo_events = get_gait_events(
    keypoint_data=stereo_data,
    kpt_labels=stereo_labels,
    gait_analysis_properties=stereo_properties,
    debug_figs_file_path=os.path.join(results_folder,"43916681.npz"),
)
stereo_results = gait_analysis(
    keypoint_data=stereo_data,
    gait_events=stereo_events,
    kpt_labels=stereo_labels,
    gait_analysis_properties=stereo_properties, 
    debug_file_path=os.path.join(results_folder,"43916681.npz")
)
stereo_result_table, stereo_result_df = display_results(stereo_results)

### Compare results

In [ ]:
qualisys_result_df.rename(columns={'back_straight': 'front_straight', 'front_straight': 'back_straight'}, inplace=True)
stereo_result_df.rename(columns={'back_straight': 'front_straight', 'front_straight': 'back_straight'}, inplace=True)

In [ ]:
qualisys_result_df=qualisys_result_df.loc[:, ['Parameter', 'Statistic', 'all','back_straight','front_straight']]
stereo_result_df = stereo_result_df.loc[:, ['Parameter', 'Statistic', 'all','back_straight','front_straight']]

In [ ]:
qualisys_result_df

In [ ]:
# difference between stereo & qualisys results (element-wise)
diff = qualisys_result_df[["all","front_straight","back_straight"]]-stereo_result_df[["all","front_straight","back_straight"]]
diff["Parameter"] = qualisys_result_df["Parameter"]
diff["Statistic"] = qualisys_result_df["Statistic"]
diff = diff[["Parameter", "Statistic", "all", "front_straight", "back_straight"]]
diff.round(3)

In [ ]:
# save all 3 tables (qualisys, stereo, diff) as csv files
for table, filename in zip([qualisys_result_df, stereo_result_df, diff],[stereo_file_path, qualisys_file_path,"diff"]):
    csv_name = os.path.join(results_folder,f"{get_basename(filename)}_{participant}.csv")
    print(csv_name)
    table.to_csv(csv_name, sep=',', encoding='utf-8', index=False, header=True)


# Statistics

## Read CSV data 

from all participants for a given task / condition

In [ ]:
parameters = [
    "step_time",
    "step_length",
    "stride_time",
    "stride_length",
    "stride_velocity",
    "swing_time",
    "double_support_time",
    "base_of_support",
]

aggregated_stereo_stats = {}

stats = ['Mean', 'CV', 'Asymmetry']
for metric in parameters:
    for stat in stats:
        inner_dict = {'Mean [m]':[],
                      'CV [%]':[],
                      'Asymmetry [%]':[]
                      }
        aggregated_stereo_stats[metric] = inner_dict

aggregated_qualisys_stats = copy.deepcopy(aggregated_stereo_stats)
statistics_results = copy.deepcopy(aggregated_stereo_stats)


In [ ]:
main_folder = "d:\\munger\\stereolabs_validation\\Code\\StereoVision\\stereo_videos\\validation_test"
# conditions = ["slow_walking","normal_walking","fast_walking"]
conditions = ["normal_walking"]

for condition in conditions:
    for g in glob.glob(f"{main_folder}\\*\\normal_walking\\results\\"):
        participant = g.split("_test\\")[1].split("\\")[0]

        stereo_csv_path = os.path.join(g,f"43916681_{participant}.csv")
        qualisys_csv_path = os.path.join(g, f"qualisys_{participant}.csv")

        if os.path.exists(stereo_csv_path) and os.path.exists(qualisys_csv_path):
            stereo_results = pd.read_csv(stereo_csv_path).to_numpy()
            qualisys_results = pd.read_csv(qualisys_csv_path).to_numpy()

        for row in stereo_results:
            # read the parameter and statistic names along with the values in the "straight_back" perspective column
            parameter, statistic = row[0:2]
            value = row[3]
            aggregated_stereo_stats[parameter][statistic].append(value)

        for row in qualisys_results:
            parameter, statistic = row[0:2]
            value = row[3]
            aggregated_qualisys_stats[parameter][statistic].append(value)

        

In [ ]:
aggregated_stereo_stats
# aggregated_qualisys_stats

In [ ]:
for parameter, statistics_dict in aggregated_qualisys_stats.items():
    for statistic, values in statistics_dict.items():
        print(parameter, statistic, len(values))

In [ ]:
parameters = list(aggregated_qualisys_stats.keys())
statistic_types = list(aggregated_qualisys_stats[parameters[0]].keys())

for parameter in parameters:
    for statistic_type in statistic_types:
        print(
            f"{parameter:<20} {statistic_type:<15} {len(aggregated_qualisys_stats[parameter][statistic_type]):3}, {len(aggregated_stereo_stats[parameter][statistic_type]):3}"
        )
        # statistics_results[parameter][statistic_type] = uniform_statistics(
        #     aggregated_qualisys_stats[parameter][statistic_type],
        #     aggregated_stereo_stats[parameter][statistic_type],
        # )

In [ ]:
uniform_statistics(aggregated_qualisys_stats['step_time']['Mean [m]'], aggregated_stereo_stats['step_time']['Mean [m]'])

In [ ]:
statistics_results

# WP

# OOP

## Data structures

In [4]:
from dataclasses import dataclass, make_dataclass
from datetime import datetime
from collections import defaultdict
from Util.gait_util import compute_asymmetry

GAIT_PARAMETERS = [
    "stride_length",
    "stride_time",
    "stride_velocity",
    "base_of_support",
    "double_support_time",
    "swing_time",
    "knee_ext_rom",
    "shoulder_flex_rom",
    "shoulder_abd_rom",
]

parameters = [(parameter_name, float, np.nan) for parameter_name in GAIT_PARAMETERS]
Parameters = make_dataclass(cls_name="Parameters", fields=parameters)


@dataclass
class GaitEvent:
    def __init__(
        self,
        type: Literal["IC", "FC"] = np.nan,
        frame: int = np.nan,
        side: Literal["left", "right"] = np.nan,
        perspective: Literal["front", "back"] = np.nan,
        segment: Literal["straight", "turn"] = np.nan,
    ):
        self.type = type
        self.frame = int(frame)
        self.side = side
        self.perspective = perspective
        self.segment = segment

    def __repr__(self):
        return f"{self.type:<10}{self.side:<10}{self.frame:<10}{self.perspective:<10}{self.segment}"

    def print(self):
        string = (
            f"{self.type:<10}{self.side:<10}{self.frame:<10}{self.perspective:<10}{self.segment}"
        )
        print(string)


@dataclass
class Flags:
    def __init__(
        self,
    ):
        self.complete_cycle = True  # all gait events are available
        self.types_ok = True  # all gait events are available & have the correct type (IC/FC)
        self.sides_ok = True  # all gait events are available & have the correct sides (right/left)
        self.perspectives_ok = (
            True  # at least IC0, IC2 are available & have the same perspective (front/back)
        )
        self.segments_ok = (
            True  # at least IC0, IC2 are available & have the same segment type (straight, turn)
        )
        self.valid = (
            True  # ge has not been deemed useless, there are no critical mistakes with the ges
        )

    def __str__(self):
        string = f"valid: {self.valid:<5}persp: {self.perspectives_ok:<5}segments: {self.segments_ok:<5}complete cycle: {self.complete_cycle:<5}types: {self.types_ok:<5}sides: {self.sides_ok:<5}"
        return string

    def __repr__(self):
        string = f"valid: {self.valid:<5}persp: {self.perspectives_ok:<5}segments: {self.segments_ok:<5}complete cycle: {self.complete_cycle:<5}types: {self.types_ok:<5}sides: {self.sides_ok:<5}"
        return string


class GaitCycle:
    def __init__(self, input_gait_events: list, keypoint_data: np.ndarray, labels: list, fps):

        self.possible_gait_events = sorted(input_gait_events, key=lambda event: event.frame)
        self.keypoints = keypoint_data
        self.labels = labels
        self.fps = fps

        # IC0, IC2 are determined at the Recording level (checks are also performed there)
        self.IC0 = self.possible_gait_events[0] if len(self.possible_gait_events) >= 2 else None
        self.IC2 = self.possible_gait_events[-1] if len(self.possible_gait_events) >= 2 else None

        self.ipsi = self.IC0.side
        self.contra = "left" if self.ipsi == "right" else "right"
        self.perspective = self.IC0.perspective
        self.segment = self.IC0.segment

        # GE priorities (based on parameters): IC0-IC2, IC1, FC0-FC1 -> select IC1 first, then FC0, FC1
        self.IC1 = self.select_gait_event(type="IC", side=self.contra)
        self.FC0 = self.select_gait_event(
            type="FC", side=self.contra, prev_event=self.IC0, next_event=self.IC1
        )
        self.FC1 = self.select_gait_event(
            type="FC", side=self.ipsi, prev_event=self.IC1, next_event=self.IC2
        )

        # get rid of not found / non-existent GEs
        self.gait_events = [
            ge for ge in [self.IC0, self.FC0, self.IC1, self.FC1, self.IC2] if ge is not None
        ]

        self.flags = Flags()
        self.set_flags()

        self.parameters = Parameters()
        if None not in [self.IC0, self.IC2]:
            self.calculate_stride_parameters()

            if self.flags.valid:
                self.calculate_knee_rom()
                self.calculate_elbow_rom()
                self.calculate_shoulder_rom()
                if None not in [self.IC1]:
                    self.calculate_base_of_support()
                    self.calculate_swing_time()
                    if None not in [self.FC0, self.FC1]:
                        self.calculate_double_support_time()

    def __str__(self):
        return f"{self.perspective:<10}{self.ipsi:<10}{self.flags.valid:<10}{[ge.frame for ge in self.gait_events]}"

    def __repr__(self):
        return f"{self.perspective:<10}{self.ipsi:<10}{self.contra:<10}{self.flags.valid:<10}{[ge.frame for ge in self.gait_events]}"

    def calculate_stride_parameters(self) -> None:  # IC0, IC2
        """
        Computes stride_time, stride_lenght and stride_velocity.
            Gait events needed: IC0, IC2

        Sets:
            parameters.stride_time
            parameters.stride_length
            parameters.stride_velocity
        """
        self.parameters.stride_time = (self.IC2.frame - self.IC0.frame) / self.fps

        # IC0_heel = self.keypoints[self.labels.index(f"{self.ipsi}_heel"), :, self.IC0.frame]
        # IC2_heel = self.keypoints[self.labels.index(f"{self.contra}_heel"), :, self.IC2.frame]
        IC0_heel = self.keypoints[self.labels.index(f"{self.ipsi}_ankle"), :, self.IC0.frame]
        IC2_heel = self.keypoints[self.labels.index(f"{self.ipsi}_ankle"), :, self.IC2.frame]
        self.parameters.stride_length = np.linalg.norm(IC2_heel - IC0_heel)

        self.parameters.stride_velocity = (
            self.parameters.stride_length / self.parameters.stride_time
            if self.parameters.stride_time >= 0
            else np.nan
        )

    def calculate_swing_time(self) -> None:  # IC1, IC2
        """
        Computes swing_time.
            Gait events needed: IC1, IC2

        Sets:
            parameters.swing_time
        """
        if self.fps > 0:
            self.parameters.swing_time = (self.IC2.frame - self.IC1.frame) / self.fps

    def calculate_double_support_time(self) -> None:  # FC0, FC1, IC0, IC1
        """
        Computes double_support_time.
            Gait events needed: FC0, FC1, IC0, IC1

        Sets:
            parameters.double_support_time
        """
        if self.fps > 0:
            self.parameters.double_support_time = (
                (self.FC0.frame - self.IC0.frame) + (self.FC1.frame - self.IC1.frame)
            ) / self.fps

    def calculate_base_of_support(self) -> None:  # IC0, IC1, IC2
        """
        Computes base_of_support.
            Gait events needed: IC0, IC1, IC2

        Sets:
            parameters.base_of_support
        """
        # using heels
        # IC1_heel = self.keypoints[self.labels.index(f"{self.contra}_heel"), :, self.IC1.frame]
        # IC0_heel = self.keypoints[self.labels.index(f"{self.ipsi}_heel"), :, self.IC0.frame]
        # IC2_heel = self.keypoints[self.labels.index(f"{self.ipsi}_heel"), :, self.IC2.frame]

        # IC2_IC0_line = IC2_heel - IC0_heel
        # IC0_IC1_line = IC0_heel - IC1_heel

        # using ankles
        IC1_ankle = self.keypoints[self.labels.index(f"{self.contra}_ankle"), :, self.IC1.frame]
        IC0_ankle = self.keypoints[self.labels.index(f"{self.ipsi}_ankle"), :, self.IC0.frame]
        IC2_ankle = self.keypoints[self.labels.index(f"{self.ipsi}_ankle"), :, self.IC2.frame]

        IC2_IC0_line = IC2_ankle - IC0_ankle
        IC0_IC1_line = IC0_ankle - IC1_ankle

        self.parameters.base_of_support = np.linalg.norm(
            np.cross(IC2_IC0_line, IC0_IC1_line)
        ) / np.linalg.norm(IC2_IC0_line)

    def calculate_knee_rom(self) -> None:  # IC0, IC2
        """
        Computes knee range of motion based on extension angle.
            Gait events needed: IC0, IC2

        Sets:
            parameters.knee_ext_rom
        """
        # TODO: figure out which approach is better
        # if it is qualisys recording use the front/back markers according to perspective
        if self.keypoints.shape[0] != 27:
            # hip_position = self.keypoints[self.labels.index(f"{self.ipsi}_hip_{self.perspective}"),:,self.IC0.frame : self.IC2.frame]
            # hip position between ipsi side front and back markers
            hip_front_position = self.keypoints[self.labels.index(f"{self.ipsi}_hip_front"),:,self.IC0.frame : self.IC2.frame]
            hip_back_position = self.keypoints[self.labels.index(f"{self.ipsi}_hip_front"),:,self.IC0.frame : self.IC2.frame]
            hip_position = np.nanmean([hip_front_position, hip_back_position], axis=0)
        # it is stereo
        else:
            hip_position = self.keypoints[
                self.labels.index(f"{self.ipsi}_hip"), :, self.IC0.frame : self.IC2.frame
            ]

        knee_position = self.keypoints[
            self.labels.index(f"{self.ipsi}_knee"), :, self.IC0.frame : self.IC2.frame
        ]
        ankle_position = self.keypoints[
            self.labels.index(f"{self.ipsi}_ankle"), :, self.IC0.frame : self.IC2.frame
        ]

        # knee ROM
        upper_leg_vector = knee_position - hip_position
        lower_leg_vector = ankle_position - knee_position

        # knee EXTENSION angle for all frames of the current gait-cycle, no projection
        knee_angles = [
            angle_between_vectors(lower_leg, upper_leg)
            for lower_leg, upper_leg in zip(lower_leg_vector.T, upper_leg_vector.T)
        ]
        # knee range of motion = max(knee angle) - min(knee angle) in given gait cycle
        self.parameters.knee_ext_rom = np.ptp(knee_angles)
        self.knee_ext_angles = knee_angles

    def calculate_elbow_rom(self) -> None:  # IC0, IC2
        """
        Computes elbow range of motion based on extension angle.
            Gait events needed: IC0, IC2

        Sets:
            parameters.elbow_ext_rom
        """
        contra_shoulder_position = self.keypoints[
            self.labels.index(f"{self.contra}_shoulder"), :, self.IC0.frame : self.IC2.frame
        ]
        elbow_position = self.keypoints[
            self.labels.index(f"{self.contra}_elbow"), :, self.IC0.frame : self.IC2.frame
        ]
        wrist_position = self.keypoints[
            self.labels.index(f"{self.contra}_wrist"), :, self.IC0.frame : self.IC2.frame
        ]

        # arm rom (elbow flexion, shoulder flexion, shoulder adduction)
        upper_arm_vector = elbow_position - contra_shoulder_position
        lower_arm_vector = wrist_position - elbow_position

        # elbow flexion angle w/o projection
        elbow_angles = [
            angle_between_vectors(upper_arm, lower_arm)
            for upper_arm, lower_arm in zip(upper_arm_vector.T, lower_arm_vector.T)
        ]
        self.parameters.elbow_ext_rom = np.ptp(elbow_angles)

    def calculate_shoulder_rom(self) -> None:  # IC0, IC2
        """
        Computes shoulder range of motion based on flexion/extension angle.
            Shoulder flexion/extension: angle between torso vector (mid_shoulder-mid_hip) and upper arm (shoulder-elbow) projections in sagittal plane.
            Shoulder adduction/abduction: angle between shoulder vector (left_shoulder-right_shoulder) and upper arm (shoulder-elbow) projections in coronal plane.
        Gait events needed: IC0, IC2

        Sets:
            parameters.shoulder_flex_rom
            parameters.shoulder_abd_rom
            self.shoulder_flex_angles
            self.shoulder_abd_angles
        """
        contra_shoulder_position = self.keypoints[
            self.labels.index(f"{self.contra}_shoulder"), :, self.IC0.frame : self.IC2.frame
        ]
        ipsi_shoulder_position = self.keypoints[
            self.labels.index(f"{self.ipsi}_shoulder"), :, self.IC0.frame : self.IC2.frame
        ]
        elbow_position = self.keypoints[
            self.labels.index(f"{self.contra}_elbow"), :, self.IC0.frame : self.IC2.frame
        ]
        mid_shoulder_position = self.keypoints[
            self.labels.index(f"mid_shoulder"), :, self.IC0.frame : self.IC2.frame
        ]

        # if it is qualisys recording use the front/back markers according to perspective
        if self.keypoints.shape[0] != 27:
            mid_hip_position = self.keypoints[
                self.labels.index(f"mid_hip_{self.perspective}"), :, self.IC0.frame : self.IC2.frame
            ]
        # it is stereo
        else:
            mid_hip_position = self.keypoints[
                self.labels.index(f"mid_hip"), :, self.IC0.frame : self.IC2.frame
            ]

        upper_arm_vector = elbow_position - contra_shoulder_position
        torso_vector = mid_hip_position - mid_shoulder_position

        sagittal_normal = contra_shoulder_position - ipsi_shoulder_position
        coronal_normal = np.cross(sagittal_normal.T, torso_vector.T)

        # shoulder flexion/extension angle with projection (torso and upper arm on sagittal plane)
        sagittal_upper_arm_proj = get_projection(sagittal_normal, upper_arm_vector)
        sagittal_torso_proj = get_projection(sagittal_normal, torso_vector)
        shoulder_flex_angles = np.array(
            [
                angle_between_vectors(upper_arm_proj, torso_proj)
                for upper_arm_proj, torso_proj in zip(sagittal_upper_arm_proj, sagittal_torso_proj)
            ]
        )
        self.parameters.shoulder_flex_rom = np.ptp(shoulder_flex_angles)
        self.shoulder_flex_angles = shoulder_flex_angles

        # shoulder abduction/adduction angle with projection (torso and upper arm on coronal plane)
        coronal_torso_proj = get_projection(coronal_normal.T, torso_vector)
        coronal_arm_proj = get_projection(coronal_normal.T, upper_arm_vector)
        shoulder_abd_angles = [
            angle_between_vectors(torso_proj, upper_arm_proj)
            for torso_proj, upper_arm_proj in zip(coronal_torso_proj, coronal_arm_proj)
        ]
        self.parameters.shoulder_abd_rom = np.ptp(shoulder_abd_angles)
        self.shoulder_abd_angles = shoulder_abd_angles

    def set_flags(self) -> None:
        """
        Checks the perspectives (front/back), segments (straight/turn), types (IC/FC), sides (left/right) of gait cycle.

        Sets:
            flags.perspectives_ok
            flags.segments_ok
            flags.types_ok
            flags.sides_ok
            flags.complete_cycle
            flags.valid
        """

        # check if perspectives are all the same (front/back)
        self.flags.perspectives_ok = False not in [
            ge.perspective == self.perspective for ge in self.gait_events
        ]
        # check if segments are all the same (straight/turn)
        self.flags.segments_ok = False not in [
            ge.segment == self.segment for ge in self.gait_events
        ]

        self.flags.types_ok = False not in [
            self.IC0.type == "IC" if self.IC0 is not None else True,
            self.FC0.type == "FC" if self.FC0 is not None else True,
            self.IC1.type == "IC" if self.IC1 is not None else True,
            self.FC1.type == "FC" if self.FC1 is not None else True,
            self.IC2.type == "IC" if self.IC2 is not None else True,
        ]
        self.flags.sides_ok = False not in [
            self.IC0.side == self.ipsi if self.IC0 is not None else True,
            self.FC0.side == self.contra if self.FC0 is not None else True,
            self.IC1.side == self.contra if self.IC1 is not None else True,
            self.FC1.side == self.ipsi if self.FC1 is not None else True,
            self.IC2.side == self.ipsi if self.IC2 is not None else True,
        ]

        # if we have all gait events
        if None not in [self.IC0, self.IC1, self.IC2, self.FC0, self.FC1]:
            # get rid of the rest (extra events)
            self.gait_events = sorted(
                [self.IC0, self.IC1, self.IC2, self.FC0, self.FC1], key=lambda event: event.frame
            )
        else:
            self.flags.complete_cycle = False

        if (
            not self.flags.types_ok
            or not self.flags.sides_ok
            or not self.flags.perspectives_ok
            or not self.flags.segments_ok
        ):
            self.flags.valid = False

    def select_gait_event(
        self,
        type: Literal["IC", "FC"],
        side: Literal["left", "right"],
        prev_event: GaitEvent = None,
        next_event: GaitEvent = None,
    ) -> GaitEvent | None:
        """
        Select the appropriate gait event from the possible_gait_events belonging to this gait cycle.

        Args:
            type: type of gait event
            side: side of gait event
            prev_event: previoius GE event (based on frame). Selected GE can only come after this
            next_event: next GE event. Selected one must come before this.

        Returns:
            selected_gait_event: the first event (ranked by frame) that satisfies the criteria, otherwise None
        """

        frame_min = prev_event.frame if prev_event is not None else self.IC0.frame
        frame_max = next_event.frame if next_event is not None else self.IC2.frame

        specified_gait_events = [
            ge
            for ge in self.possible_gait_events
            if (
                ge.type == type
                and ge.side == side
                and ge.perspective == self.perspective
                and ge.segment == self.segment
                and frame_min <= ge.frame <= frame_max
            )
        ]
        selected_gait_event = next(iter(specified_gait_events), None)
        return selected_gait_event

    def print(self):
        string = f"{self.perspective:<10}{self.ipsi:<10}{self.contra:<10}{self.valid:<10}{[ge.frame for ge in self.gait_events]}   {self.parameters}"
        print(string)


class Recording:
    def __init__(
        self,
        keypoints: np.ndarray,
        labels,
        recording_type: Literal["stereo", "qualisys"],
        gait_analysis_properties: dict,
        debug_folder: str = None,
    ):
        self.keypoints = keypoints
        self.labels = labels
        self.recording_type = recording_type

        # gait analysis properties
        gait_analysis_properties = gait_analysis_properties[self.recording_type]
        self.fps = gait_analysis_properties["fps"]
        self.max_gap = gait_analysis_properties["max_gap_interpolation"]
        self.filter_cutoff = gait_analysis_properties["filter_cutoff"]
        self.filter_order = gait_analysis_properties["filter_order"]

        self.heel_thr = gait_analysis_properties["heel_thr"]
        self.toe_thr = gait_analysis_properties["toe_thr"]

        self.min_stride_time = gait_analysis_properties["stride_time_min"] * self.fps
        self.max_stride_time = gait_analysis_properties["stride_time_max"] * self.fps
        self.min_swing_time = gait_analysis_properties["swing_time_min"] * self.fps
        self.min_stance_time = gait_analysis_properties["stance_time_min"] * self.fps

        if not os.path.exists(debug_folder):
            os.mkdir(debug_folder)
        self.debug_folder = debug_folder

        self.turn_mask, self.perspective = self.get_turns_and_perspective()

        if self.perspective[0] != "front":
            self.keypoints[:, 1, :] = self.keypoints[:, 1, :] * (-1)
            self.turn_mask, self.perspective = self.get_turns_and_perspective()

        self.initial_gait_events = self.get_gait_events(velocity=1, create_debug_fig=False)
        self.initial_gait_cycles = self.get_gait_cycles(self.initial_gait_events)
        self.velocity_estimate = np.nanmean(
            [
                gc.parameters.stride_velocity
                for gc in self.initial_gait_cycles
                if gc.flags.valid
                # and gc.perspective == "back"
            ]
        )  # TODO: is back perspective the only valid way?
        self.gait_events = self.get_gait_events(
            velocity=self.velocity_estimate, create_debug_fig=True
        )
        self.gait_cycles = self.get_gait_cycles(self.gait_events)

        if self.debug_folder is not None:
            self.create_gait_cycles_figure()

    def get_gait_cycles(self, gait_events: list):
        gait_cycles = []
        for event in gait_events:
            if event.type == "IC":
                next_ipsi_IC = next(
                    iter(
                        [
                            ge
                            for ge in gait_events
                            if (
                                ge.type == "IC"
                                and ge.side == event.side
                                and ge.frame > event.frame
                                and ge.frame <= event.frame + self.max_stride_time
                                and ge.perspective == event.perspective
                                and ge.segment == event.segment
                            )
                        ]
                    ),
                    None,
                )
                if next_ipsi_IC is not None:
                    possible_events = [
                        ge
                        for ge in gait_events
                        if (ge.frame >= event.frame and ge.frame <= next_ipsi_IC.frame)
                    ]
                    current_gait_cycle = GaitCycle(
                        possible_events, self.keypoints, self.labels, self.fps
                    )
                    gait_cycles.append(current_gait_cycle)

        return gait_cycles

    def get_turns_and_perspective(
        self,
        min_peak_height: float = 0.3,
        relative_height: float = 0.85,
        min_walk_duration: float = 2,
    ) -> tuple:
        """
        Identifies turning segments (straight/turning) and perspective (front / back) based on shoulder coordinates using peak detection method.

        Args:
            min_peak_height:    only peaks above this height will be considered
            relative_height:    height at which the bases of a peak will be taken. It is MEASURED FROM APEX
            min_walk_duration:  min elapsed time (in seconds) between turns (1 turn = 180 deg)

        Returns:
            turn_mask: boolean array indicating straight turning segments (1) and straight segments (0).

            perspective: bollean array indicating frontal facing (1) and back facing (0) segments.
        """
        # get the y coord of shoulders (for qualisys x-y plane is horizontal, y coord was forwards/backwards movement)
        shoulder_R = self.keypoints[self.labels.index("right_shoulder"), 1, :]
        shoulder_L = self.keypoints[self.labels.index("left_shoulder"), 1, :]

        # normalize signal
        shoulder_diff = np.abs(np.diff(shoulder_R - shoulder_L))
        shoulder_diff = shoulder_diff / np.nanmax(shoulder_diff)

        # Interpolate NaNs in shoulder_diff
        if np.any(np.isnan(shoulder_diff)):
            nans = np.isnan(shoulder_diff)
            shoulder_diff[nans] = np.interp(
                np.flatnonzero(nans), np.flatnonzero(~nans), shoulder_diff[~nans]
            )

        # min distance from peak to peak (2 seconds of straight movement) in samples
        min_peak_distance = min_walk_duration * self.fps

        peaks, peak_properties = find_peaks(
            shoulder_diff, height=min_peak_height, distance=min_peak_distance
        )
        widths, width_heights, left_ips, right_ips = peak_widths(
            x=shoulder_diff, peaks=peaks, rel_height=relative_height
        )

        # create mask to indicate straight walking segments (straight=False, turn=True)
        turn_mask = np.zeros_like(shoulder_R)
        for left_base_idx, right_base_idx in zip(
            np.floor(left_ips).astype(int), np.ceil(right_ips).astype(int)
        ):
            # turn_segment_length = right_base_idx - left_base_idx
            turn_mask[left_base_idx:right_base_idx] = 1

        # create mask to indicate front vs back perspectives (front=True, back=False)
        perspective = np.where(shoulder_L > shoulder_R, "front", "back")
        turn_mask = np.where(turn_mask == 1, "turn", "straight")

        # PLOT
        if self.debug_folder is not None:

            t = np.linspace(0, len(shoulder_R) / self.fps, len(shoulder_R))

            plt.close("all")
            fig, axs = plt.subplots(3, 1, figsize=(14, 5))

            axs[0].plot(t[1:], shoulder_diff, label="shoulder diff")
            axs[0].plot(left_ips / self.fps, width_heights, "o", label="turn start", markersize=4)
            axs[0].plot(right_ips / self.fps, width_heights, "o", label="turn end", markersize=4)
            axs[0].plot(t, turn_mask, label="straight/turn")
            axs[0].set_title("turn mask")

            axs[1].plot(t, shoulder_L, label="left")
            axs[1].plot(t, shoulder_R, label="right")
            axs[1].set_title("shoulder position (horizontal)")

            axs[2].plot(t, perspective, label="perspective")
            axs[2].set_xlabel("Time [s]")
            axs[2].set_title("Perspective")

            for ax in axs[:-1]:
                ax.legend(loc="upper left")
                ax.grid()

            plt.tight_layout()

            out_filename = os.path.join(
                self.debug_folder, f"{self.recording_type}_turns_and_perspective.png"
            )
            plt.savefig(out_filename)
            plt.close("all")
            print(f"T&P figure saved to: \n{out_filename}")

        return (turn_mask, perspective)

    def get_gait_events(self, velocity, create_debug_fig: bool = False):
        heel_thr = self.heel_thr * velocity
        ankle_thr = self.heel_thr * velocity
        big_toe_thr = self.toe_thr * velocity

        all_events = []

        for side in ["left", "right"]:
            heel = self.keypoints[self.labels.index(f"{side}_heel"), :, :]
            big_toe = self.keypoints[self.labels.index(f"{side}_big_toe"), :, :]
            ankle = self.keypoints[self.labels.index(f"{side}_ankle"), :, :]

            heel_vel = (
                np.linalg.norm(np.diff(heel, axis=1, prepend=heel[:, [0]]), axis=0) * self.fps
            )
            ankle_vel = (
                np.linalg.norm(np.diff(ankle, axis=1, prepend=ankle[:, [0]]), axis=0) * self.fps
            )
            big_toe_vel = (
                np.linalg.norm(np.diff(big_toe, axis=1, prepend=big_toe[:, [0]]), axis=0) * self.fps
            )

            # ground_contact_thrs = (
            #     (heel_vel < heel_thr) | (ankle_vel < ankle_thr) | (big_toe_vel < big_toe_thr)
            # ).astype(int)
            ground_contact_thrs = (ankle_vel < ankle_thr).astype(int)

            # remove events from turn segments
            ground_contact_straight = np.where(self.turn_mask == "straight", ground_contact_thrs, 1)

            contact_diff_straight = np.diff(
                ground_contact_straight, prepend=ground_contact_straight[0]
            )

            # remove too short swing periods
            ground_conact_no_short_swing = np.copy(ground_contact_straight)
            starts = np.nonzero(contact_diff_straight == -1)[0]
            ends = np.nonzero(contact_diff_straight == 1)[0]

            for start_frame, end_frame in zip(starts, ends):
                if end_frame - start_frame < self.min_swing_time:
                    ground_conact_no_short_swing[start_frame:end_frame] = 1

            contact_diff_no_short_swing = np.diff(
                ground_conact_no_short_swing, prepend=ground_conact_no_short_swing[0]
            )

            # remove too short contact periods
            ground_contact_no_short_contact = np.copy(ground_conact_no_short_swing)
            starts = np.nonzero(contact_diff_no_short_swing == 1)[0]
            ends = np.nonzero(contact_diff_no_short_swing == -1)[0]
            for idx, [start_frame, end_frame] in enumerate(zip(starts, ends)):
                if idx + 1 < ends.shape[0]:
                    if ends[idx + 1] - start_frame < self.min_stance_time:
                        ground_contact_no_short_contact[start_frame : ends[idx + 1]] = 0

            contact_diff_no_short_contact = np.diff(
                ground_contact_no_short_contact, prepend=ground_contact_no_short_contact[0]
            )

            IC_frames = np.nonzero(contact_diff_no_short_contact == 1)[0]
            FC_frames = np.nonzero(contact_diff_no_short_contact == -1)[0]

            IC_events = [
                GaitEvent(
                    type="IC",
                    frame=frame,
                    side=side,
                    perspective=self.perspective[frame],
                    segment=self.turn_mask[frame],
                )
                for frame in IC_frames
                if frame > 100
            ]

            FC_events = [
                GaitEvent(
                    type="FC",
                    frame=frame,
                    side=side,
                    perspective=self.perspective[frame],
                    segment=self.turn_mask[frame],
                )
                for frame in FC_frames
                if frame < len(self.turn_mask)
            ]

            all_events.extend(IC_events + FC_events)

            # PLOT
            if create_debug_fig:
                plt.close("all")
                fig, axs = plt.subplots(4, 1, figsize=(14, 7))

                axs[0].plot(ankle_vel, label="velocity")
                axs[0].plot(ground_contact_thrs * ankle_thr, label="ground contact at thr")
                # axs[0].plot(self.turn_mask,'r--')

                axs[1].plot(ground_contact_thrs + 6, label="raw", color="#ff8800ff")
                axs[1].plot(ground_contact_straight + 4, label="no turns", color="#ffd000ff")
                axs[1].plot(
                    ground_conact_no_short_swing + 2, label="no short swings", color="#d9e717"
                )
                axs[1].plot(
                    ground_contact_no_short_contact, label="no short contacts", color="#6dc916"
                )

                axs[2].plot(contact_diff_straight + 4, label="no turns", color="#ffd000ff")
                axs[2].plot(
                    contact_diff_no_short_swing + 2, label="no short swings", color="#d9e717"
                )
                axs[2].plot(
                    contact_diff_no_short_contact, label="no short contact", color="#6dc916"
                )

                axs[3].plot(self.perspective, label="perspective")

                # fmt: off
                axs[0].set_title("ankle velocity")
                axs[1].set_title("ground contact")
                axs[2].set_title("ground contact diff (GEs)")

                axs[0].set_yticks([0, 3, ankle_thr])
                axs[0].set_ylim([0,3])
                # fmt: on
                for ax in axs:
                    ax.legend(loc="upper left")
                    ax.grid()

                plt.tight_layout()
                out_filename = os.path.join(
                    self.debug_folder, f"{self.recording_type}_gait_events_{side}.png"
                )
                plt.savefig(out_filename)
                plt.close("all")
                print(f"GE figure saved to: \n{out_filename}")
        all_events_sorted = sorted(all_events, key=lambda event: event.frame)

        return all_events_sorted

    def create_gait_cycles_figure(self):
        if self.debug_folder is not None:
            plt.close("all")
            fig, axs = plt.subplots(3, 1, figsize=(14, 5))

            # show GEs of gait cycles
            for gc in self.gait_cycles:
                alpha = 1 if gc.flags.valid else 0.25
                if gc.IC0 != None:
                    axs[0].vlines(gc.IC0.frame, 0, 0.9, "c", linewidth=3, alpha=alpha)
                if gc.IC2 != None:
                    axs[0].vlines(gc.IC0.frame, 1.1, 1.9, "m", linewidth=3, alpha=alpha)
                if gc.IC1 != None:
                    axs[0].vlines(gc.IC0.frame, 2.1, 2.9, "r", linewidth=3, alpha=alpha)
                if gc.FC1 != None:
                    axs[0].vlines(gc.IC0.frame, 3.1, 3.9, "b", linewidth=3, alpha=alpha)
                if gc.FC0 != None:
                    axs[0].vlines(gc.IC0.frame, 4.1, 4.9, "g", linewidth=3, alpha=alpha)

            axs[0].set_yticks([0.5, 1.5, 2.5, 3.5, 4.5], ["IC0", "IC2", "IC1", "FC1", "FC0"])
            axs[0].set_xlim([0, len(self.perspective)])
            axs[0].set_title("gait cycle composition")

            # show flags of gait cycles
            for gc in self.gait_cycles:
                if gc.flags.valid:
                    axs[1].vlines(gc.IC0.frame, 0, 0.9, "gray", linewidth=3)
                if gc.flags.perspectives_ok:
                    axs[1].vlines(gc.IC0.frame, 1.1, 1.9, "gray", linewidth=3)
                if gc.flags.segments_ok:
                    axs[1].vlines(gc.IC0.frame, 2.1, 2.9, "gray", linewidth=3)
                if gc.flags.types_ok:
                    axs[1].vlines(gc.IC0.frame, 3.1, 3.9, "gray", linewidth=3)
                if gc.flags.sides_ok:
                    axs[1].vlines(gc.IC0.frame, 4.1, 4.9, "gray", linewidth=3)

            axs[1].set_yticks(
                [0.5, 1.5, 2.5, 3.5, 4.5], ["valid", "persp", "segments", "types", "sides"]
            )
            axs[1].set_xlim([0, len(self.perspective)])
            axs[1].set_title("gait cycle flags")

            axs[2].plot(self.perspective)
            axs[2].plot(self.turn_mask)
            axs[2].set_xlim([0, len(self.perspective)])

            plt.tight_layout()
            # plt.show()

            out_filename = os.path.join(self.debug_folder, f"{self.recording_type}_gait_cycles.png")
            plt.savefig(out_filename)
            print(f"Gait cycles figure saved to: \n{out_filename}")


class RecordingPair:
    def __init__(self, qualisys_file_path, stereo_file_path, debug_folder_path):
        self.qualisys_file_path = qualisys_file_path
        self.stereo_file_path = stereo_file_path
        self.debug_folder = debug_folder_path

        qualisys_keypoints, qualisys_labels = load_keypoints_and_labels(qualisys_file_path)
        stereo_keypoints, stereo_labels = load_keypoints_and_labels(stereo_file_path)

        self.qualisys = Recording(
            keypoints=qualisys_keypoints,
            labels=qualisys_labels,
            recording_type="qualisys",
            gait_analysis_properties=gait_analysis_properties,
            debug_folder=debug_folder_path,
        )
        self.stereo = Recording(
            keypoints=stereo_keypoints,
            labels=stereo_labels,
            recording_type="stereo",
            gait_analysis_properties=gait_analysis_properties,
            debug_folder=debug_folder_path,
        )
        self.get_sync_pairs(window=70)
        self.create_sync_figure()

        self.statistics_results = self.create_result_container()
        self.get_statistics_results()

    def get_sync_pairs(self, window: int = 60):
        gc_pairs = []
        shift = self.stereo.gait_events[0].frame - self.qualisys.gait_events[0].frame

        for gc in self.stereo.gait_cycles:
            if gc.flags.valid:
                sync_frame = gc.IC0.frame - shift

                potential_pairs = [
                    p
                    for p in self.qualisys.gait_cycles
                    if abs(p.IC0.frame - sync_frame) < window
                    and p.flags.valid
                    and p.ipsi == gc.ipsi
                    and p.perspective == gc.perspective
                    and p.IC0.frame + shift + 10 >= gc.IC0.frame
                ]
                if len(potential_pairs) != 0:
                    gc_pairs.append([gc, potential_pairs[0]])

        self.sync_pairs = np.asarray(gc_pairs)
        self.sync_shift = shift

    def create_sync_figure(self):
        valid_qualisys_gcs = [gc.IC0.frame for gc in self.qualisys.gait_cycles if gc.flags.valid]
        qualisys_gcs_vis = np.zeros(self.qualisys.keypoints.shape[2])
        qualisys_gcs_vis[valid_qualisys_gcs] = 1

        valid_stereo_gcs = [gc.IC0.frame for gc in self.stereo.gait_cycles if gc.flags.valid]
        stereo_gcs_vis = np.zeros(self.stereo.keypoints.shape[2])
        stereo_gcs_vis[valid_stereo_gcs] = 1

        plt.close("all")
        fig, axs = plt.subplots(2, 1, figsize=(14, 3))

        axs[0].plot(stereo_gcs_vis[self.sync_shift :], color="#1f77b4")
        axs[0].plot(self.stereo.turn_mask[self.sync_shift :], color="#ff7f0e", alpha=0.5)

        axs[1].plot(qualisys_gcs_vis, color="#1f77b4")
        axs[1].plot(self.qualisys.turn_mask, color="#ff7f0e", alpha=0.5)

        for stereo_gc, qualisys_gc in self.sync_pairs:
            stereo_frame = stereo_gc.IC0.frame
            qualisys_frame = qualisys_gc.IC0.frame

            axs[0].plot(stereo_frame - self.sync_shift, 0.5, marker="o")
            axs[1].plot(qualisys_frame, 0.5, marker="o")

        axs[0].set_title("stereo")
        axs[1].set_title("qualisys")

        axs[0].set_xlim([0, self.qualisys.keypoints.shape[2]])
        axs[1].set_xlim([0, self.qualisys.keypoints.shape[2]])

        plt.tight_layout()
        out_filename = os.path.join(self.debug_folder, f"sync.png")
        plt.savefig(out_filename)
        print(f"Sync figure saved to: \n{out_filename}")

    def create_result_container(self):
        metrics = {"mean": np.nan, "std": np.nan, "cv": np.nan, "asymmetry": np.nan}

        # statistics_results = {}
        # for perspective in ["front","back"]:

        #     recording_results = {}
        #     for recording_type in ["qualisys", "stereo"]:
        #         parameter_results = {}
        #         for parameter in GAIT_PARAMETERS:
        #             parameter_results[parameter] = {}
        #         recording_results[recording_type] = parameter_results
        #     statistics_results[perspective] = recording_results

        # return statistics_results
        nested_dict = lambda: defaultdict(nested_dict)
        statistics_results = nested_dict()
        for perspective in ["front", "back"]:
            for recording_type in ["qualisys", "stereo"]:
                for parameter in GAIT_PARAMETERS:
                    statistics_results[perspective][recording_type][parameter] = {}
        return statistics_results

    def get_statistics_results(self):
        """
        Calculates metrics for stereo and qualisys gait cycles present in self.sync_pairs for different perspectives.
        """
        for perspective in ["front", "back"]:
            for parameter in GAIT_PARAMETERS:
                for type_idx, recording_type in enumerate(["stereo", "qualisys"]):

                    values_left = [
                        getattr(gc.parameters, parameter, np.nan)
                        for gc in self.sync_pairs[:, type_idx]
                        if gc.perspective == perspective and gc.ipsi == "left"
                    ]
                    values_right = [
                        getattr(gc.parameters, parameter, np.nan)
                        for gc in self.sync_pairs[:, type_idx]
                        if gc.perspective == perspective and gc.ipsi == "right"
                    ]

                    values_all = np.concat((values_left, values_right))
                    mean = np.nanmean(values_all)
                    std = np.nanstd(values_all)
                    cv = 100 * std / mean if mean != 0 else np.nan
                    asymmetry = compute_asymmetry(values_left, values_right)

                    metrics = {"mean": mean, "std": std, "cv": cv, "asymmetry": asymmetry}

                    self.statistics_results[perspective][recording_type][parameter] = metrics

    def print_statistics_results(self):
        stat = self.statistics_results
        rows = []
        for (parameter_name, qualisys_front), stereo_front, qualisys_back, stereo_back in zip(
            stat["front"]["qualisys"].items(),
            stat["front"]["stereo"].values(),
            stat["back"]["qualisys"].values(),
            stat["back"]["stereo"].values(),
        ):
            for (
                (metric_name, qualisys_front_value),
                stereo_front_value,
                qualisys_back_value,
                stereo_back_value,
            ) in zip(
                qualisys_front.items(),
                stereo_front.values(),
                qualisys_back.values(),
                stereo_back.values(),
            ):
                rows.append(
                    [
                        parameter_name,
                        metric_name,
                        qualisys_front_value,
                        stereo_front_value,
                        qualisys_back_value,
                        stereo_back_value,
                    ]
                )
        pd.options.display.float_format = "{:,.1f}".format
        df = pd.DataFrame(
            rows, columns=["PARAMETER", "METRIC", "Q FRONT", "S FRONT", "Q BACK", "S BACK"]
        )
        print(df)


def load_keypoints_and_labels(npz_data_path: str):
    # load keypoint data and labels
    if os.path.exists(npz_data_path):
        loaded_data = np.load(npz_data_path)
        keypoints = loaded_data["keypoints"]
        labels = loaded_data["labels"].tolist()
        print(f"loaded data from: {npz_data_path} with keys: {[key for key in loaded_data.keys()]}")
    else:
        raise FileNotFoundError(f"File: {npz_data_path} does not exist!")
    return keypoints, labels

In [ ]:
qualisys_file_path = f"./stereo_videos\\validation_test\\{participant}\\{paradigm}\\qualisys.npz"
stereo_file_path = f"./stereo_videos\\validation_test\\{participant}\\{paradigm}\\stereo.npz"
debug_folder_path = f"./stereo_videos\\validation_test\\{participant}\\{paradigm}\\Results\\"

In [ ]:
stereo_file_path = f".\\filter_test_data\\stereo_butter.npz"
qualisys_file_path = f".\\filter_test_data\\qualisys.npz"
debug_folder_path = f".\\filter_test_data"

In [ ]:
Pair = RecordingPair(
    qualisys_file_path=qualisys_file_path,
    stereo_file_path=stereo_file_path,
    debug_folder_path=debug_folder_path
)

In [ ]:
Pair.print_statistics_results()

In [ ]:
# PLOT KNEE ANGLES
plt.close('all')
fig, axs = plt.subplots(2,2, figsize=(14,6))
for ax_idx, perspective in enumerate(["front", "back"]):
    stereo_angles = [gc.knee_ext_angles for gc in Pair.stereo.gait_cycles if gc.flags.valid and gc.perspective==perspective]
    qualisys_angles = [gc.knee_ext_angles for gc in Pair.qualisys.gait_cycles if gc.flags.valid and gc.perspective==perspective]
    for stereo, qualisys in zip(stereo_angles, qualisys_angles) :
        axs[ax_idx,0].plot(stereo)
        axs[ax_idx,1].plot(qualisys)
        axs[ax_idx][0].set_title(perspective)
        axs[ax_idx][1].set_title(perspective)
        axs[ax_idx][0].set_ylim([0,80])
        axs[ax_idx][1].set_ylim([0,80])
        axs[ax_idx][0].grid()
        axs[ax_idx][1].grid()

plt.tight_layout()


In [ ]:
# PLOT SHOULDER FLEX ANGLES
plt.close('all')
fig, axs = plt.subplots(2,2, figsize=(14,6))
for ax_idx, perspective in enumerate(["front", "back"]):
    stereo_angles = [gc.shoulder_flex_angles for gc in Pair.stereo.gait_cycles if gc.flags.valid and gc.perspective==perspective]
    qualisys_angles = [gc.shoulder_flex_angles for gc in Pair.qualisys.gait_cycles if gc.flags.valid and gc.perspective==perspective]
    for stereo, qualisys in zip(stereo_angles, qualisys_angles):
        axs[ax_idx,0].plot(stereo)
        axs[ax_idx,1].plot(qualisys)
        axs[ax_idx][0].set_title(perspective)
        axs[ax_idx][1].set_title(perspective)
        # axs[ax_idx][0].set_ylim([0,80])
        # axs[ax_idx][1].set_ylim([0,80])
        axs[ax_idx][0].grid()
        axs[ax_idx][1].grid()

plt.tight_layout()


## recording statistics

In [ ]:
print(len(Pair.sync_pairs))
print(len([gc for gc in Pair.stereo.gait_cycles if gc.flags.valid]))
print(len([gc for gc in Pair.qualisys.gait_cycles if gc.flags.valid]))

In [ ]:
valid_gait_cycles = [gc for gc in Pair.qualisys.gait_cycles if gc.flags.valid]
print(valid_gait_cycles[0].parameters)
print("\n")

In [ ]:
from collections import defaultdict

def compute_asymmetry_new(left_values, right_values):
    if not (np.isnan(left_values).all() or np.isnan(right_values).all()):
        left_mean = np.nanmean(left_values)
        right_mean = np.nanmean(right_values)
        smaller = np.nanmin([left_mean, right_mean])
        larger = np.nanmax([left_mean, right_mean])
        return 100 * (1 - smaller / larger) if larger > 0 else np.nan

def create_result_container():
    nested_dict = lambda: defaultdict(nested_dict)
    statistics_results = nested_dict()
    for perspective in ["front", "back"]:
        for recording_type in ["qualisys", "stereo"]:
            for parameter in GAIT_PARAMETERS:
                statistics_results[perspective][recording_type][parameter] = {}
    return statistics_results

statistics_results = create_result_container()

for perspective in ["front", "back"]:
    print(perspective)
    for parameter in GAIT_PARAMETERS:
        for type_idx, recording_type in enumerate(["stereo", "qualisys"]):

            values_left = [
                getattr(gc.parameters, parameter, np.nan)
                for gc in Pair.sync_pairs[:, type_idx]
                if gc.perspective == perspective and gc.ipsi == "left"
            ]
            values_right = [
                getattr(gc.parameters, parameter, np.nan)
                for gc in Pair.sync_pairs[:, type_idx]
                if gc.perspective == perspective and gc.ipsi == "right"
            ]

            values_all = np.concat((values_left, values_right))
            mean = np.nanmean(values_all)
            std = np.nanstd(values_all)
            cv = 100 * std / mean if mean != 0 else np.nan
            asymmetry = compute_asymmetry_new(values_left, values_right)

            metrics = {"mean": mean, "std": std, "cv": cv, "assymetry": asymmetry}

            statistics_results[perspective][recording_type][parameter] = metrics

print(statistics_results)

In [ ]:
def create_result_container():    
    for type in ["qualisys","stereo"]:
        for parameter in GAIT_PARAMETERS:
            for metric in ["mean", "cv", "asymmetry"]:
                pair_metrics = {"qualisys": {}, "stereo": {}}
                for type in ["qualisys", "stereo"]:
                    for parameter in GAIT_PARAMETERS:
                        pair_metrics[type][parameter] = {}
                        for metric in ["mean", "std", "cv", "asymmetry"]:
                            pair_metrics[type][parameter][metric] = np.nan
    result_container = {}
    for perspective in ["front", "back"]: 
        result_container[perspective] = pair_metrics
    return result_container

In [ ]:
from collections import defaultdict


def create_result_container():
    nested_dict = lambda: defaultdict(nested_dict)

    statistics_results = nested_dict()
    # target_dict[key1][key2] = val
    for perspective in ["front","back"]:
        for recording_type in ["qualisys","stereo"]:
            for parameter in GAIT_PARAMETERS:
                for metric in ["mean", "std", "cv", "asymmetry"]:
                    statistics_results[perspective][recording_type][parameter][metric] = np.nan

    return statistics_results

In [ ]:
from collections import defaultdict
nested_dict = lambda: defaultdict(nested_dict)

target_dict = nested_dict()
for letter in ["a","b","c"]:
    for number in ["one","two"]:
        for stuff in ["this","that"]:
            for thing in ["asdf","fdsa"]:
                target_dict[letter][number][stuff][thing] = 0
target_dict['a']['one']['that']['asdf']

In [ ]:
metrics = {'mean': np.nan,'std': np.nan,'cv': np.nan,'asymmetry': np.nan} 

stat_results = {}
for perspective in ["front","back"]:
    type_results = {"qualisys":{},"stereo":{}}
    for type in ["qualisys","stereo"]:
        for parameter in GAIT_PARAMETERS:
            type_results[type][parameter] = metrics
    stat_results[perspective] = type_results

stat_results

## check gait cycles

In [ ]:
fig= plt.figure(figsize=(14,3))

gait_events = Pair.stereo.gait_events

for ge in gait_events:
    alpha = 1 if ge.segment=='straight' else 0.2
    color = "#55e05c" if ge.type=='IC' else "#5565e0"
    match ge.side:
        case 'left':
            plt.vlines(x=ge.frame, ymin=0, ymax=1, alpha=alpha, colors=color)
        case 'right':
            plt.vlines(x=ge.frame, ymin=-1, ymax=0, alpha=alpha, colors=color)

plt.tight_layout()

In [ ]:
for gc_idx, gc in enumerate(Pair.stereo.gait_cycles):
    print(gc_idx)
    print(gc)
    print(gc.flags)
    for ge, pge in zip(gc.gait_events, gc.possible_gait_events):
        print(f"{ge} | {pge}")
    print(f"IC0: {gc.IC0}")
    print(f"FC0: {gc.FC0}")
    print(f"IC1: {gc.IC1}")
    print(f"FC1: {gc.FC1}")
    print(f"IC2: {gc.IC2}")
    
    print('')
    print('')

### Bokeh

In [ ]:
tuple(np.random.randint(low=0, high=255,size=(3)))

In [ ]:
color = LinearColorMapper(palette = "Magma256", low = 0, high = 1)
color.

In [ ]:
# importing the modules 
import itertools
import bokeh
from bokeh.plotting import figure, output_file, show 
from bokeh.models import LinearColorMapper

bokeh.io.output_notebook()

# file to save the model 
# output_file("gfg.html") 
# colors = itertools.cycle(Category10[len(Pair.stereo.gait_cycles)])

color = LinearColorMapper(palette = "Magma256", low = 0, high = 10)
# instantiating the figure object 
graph = figure(title = "Bokeh Hiding Glyphs",sizing_mode="stretch_width", max_width=1500, height=250) 
graph.graph
for gc_idx, gc in enumerate(Pair.stereo.gait_cycles):
    # color = tuple(np.random.randint(low=0, high=100,size=(3)))
    for ge in gc.possible_gait_events:
        graph.vbar(x=ge.frame, top=1, width=1, legend_label=f"{gc_idx}", muted_alpha=0.2)

# for ge in Pair.stereo.gait_events:
#     graph.vbar(x=ge.frame, top=1, width=1, color="violet", legend_label=f"ic", muted_alpha=0.2)


# enable hiding of the glyphs 
graph.legend.click_policy = "mute"

# displaying the model 
show(graph)

## stereo data processing

In [14]:
from OneEuroFilter import OneEuroFilter
import time


def get_2d_keypoints(video_file_path: str, labels: list) -> np.ndarray:
    """
    Preform 2D pose estimation on RGB video, return x,y pixel coordinates of predicted keypoints.

    Args:
        video_file_path: path of input video
        labels: labels used for indexing keypoints array

    Returns:
        keypoints_2d: array containing the calculated keypoint coordinates. (frames x num_kpts x (x,y))

    """
    # create custom detector & predictor model
    custom_pose_model = load_pose_model()

    print(f"processing: {video_file_path}")

    # containers to store the results in
    keypoints_2d = []
    mid_shoulder = []

    cap = cv2.VideoCapture(video_file_path)
    if cap.isOpened() == False:
        print(f"Error opening video file: {video_file_path}")

    total_num_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    processed_num_frames = 0
    start_time = time.time_ns()

    while cap.isOpened():
        ret, frame = cap.read()

        if ret == True and frame is not None:
            processed_num_frames += 1
            width = frame.shape[1]
            height = frame.shape[0]

            # inference
            keypoints, scores = custom_pose_model(frame)

            # visualize
            boxes = [pose_to_bbox(x) for x in keypoints]
            img_show = draw_bbox(frame, boxes, (0, 0, 255))

            # store results (frames * kpt * (x,y)), remove multiperson cases
            if keypoints.shape != (1, 26, 2):
                keypoints = keypoints[0, :, :]
                scores = scores[0, :]

            keypoints = np.squeeze(keypoints)

            # add mid_shoulder keypoint
            left_shouder_position = keypoints[labels.index("left_shoulder")]
            right_shouder_position = keypoints[labels.index("right_shoulder")]
            mid_shoulder_position = np.nanmean(
                [left_shouder_position, right_shouder_position], axis=0
            )

            keypoints_2d.append(keypoints)
            mid_shoulder.append(mid_shoulder_position)

            for kpt in keypoints[:, :]:
                cv2.circle(img_show, (int(kpt[0]), int(kpt[1])), 3, (0, 255, 0), 2)

            cv2.imshow("Image", cv2.resize(img_show, (int(width / 1.5), int(height / 1.5))))

            # display how much time elapsed/is left
            clear_output(wait=True)
            end_time = time.time_ns()
            elapsed_time = (end_time - start_time) / 1000000000

            if int(elapsed_time % 5) == 0:

                fps = round(processed_num_frames / elapsed_time, 2)
                done_ratio = processed_num_frames / total_num_frames
                expected_duration_min = elapsed_time / (done_ratio * 60)

                display(
                    f"exp_duration: {round(expected_duration_min,2)} minutes, elapsed: {round(elapsed_time/60,2)} minutes"
                )
                display(f"done: {round(done_ratio*100,2)}%, fps: {fps}")

            # Press Q on keyboard to exit
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break
        else:
            break
    cap.release()
    cv2.destroyAllWindows()

    keypoints_2d = np.asarray(keypoints_2d)
    mid_shoulder = np.expand_dims(np.array(mid_shoulder), axis=1)
    keypoints_2d = np.hstack((keypoints_2d, mid_shoulder))

    print(f"shape of keypoints_2d:  {keypoints_2d.shape} (frames * kpt * (x, y))")

    return keypoints_2d


def create_filters(
    num_keypoints: int, freq: float = 30.0, min_cutoff: float = 1.0, beta: float = 0.0
) -> list[OneEuroFilter]:
    """
    Create a list of OneEuroFilters for each dimension of a keypoint (x,y,z).

    Args:
        num_keypoints: number of keypoints
        freq: frequency of the time series
        min_cutoff: min cutoff frequency in Hz
        beta: parameter to reduce latency

    Returns:
        filters: list of OneEuroFilters (num_keypoints,3)
    """
    # Create filters for smoothing keypoints
    filters = []
    for _ in range(num_keypoints):
        filters.append(
            {
                "x": OneEuroFilter(freq, min_cutoff, beta),
                "y": OneEuroFilter(freq, min_cutoff, beta),
                "z": OneEuroFilter(freq, min_cutoff, beta),
            }
        )
    return filters


def filter_2d_keypoints_euro(keypoints_2d: np.ndarray, freq: float, min_cutoff: float, beta: float):
    """
    Filters 2D keypoints using OneEuroFilters. Shape remains the same.
    """
    one_euro_filters = create_filters(
        num_keypoints=keypoints_2d.shape[1], freq=freq, min_cutoff=min_cutoff, beta=beta
    )
    keypoints_2d_filtered = []

    for keypoints_in_frame in keypoints_2d:
        filtered_keypoints_in_frame = []

        for kpt_idx, [x, y] in enumerate(keypoints_in_frame):
            filtered_x = one_euro_filters[kpt_idx]["x"](x)
            filtered_y = one_euro_filters[kpt_idx]["y"](y)
            filtered_keypoints_in_frame.append([filtered_x, filtered_y])

        keypoints_2d_filtered.append(filtered_keypoints_in_frame)
    keypoints_2d_filtered = np.asarray(keypoints_2d_filtered)
    return keypoints_2d_filtered


def filter_2d_keypoints_butter(
    keypoints_2d: np.ndarray, order: int = 4, lowcut: float = 2, freq: float = 60
):
    """
    Filter 2D keypoints using butterworth lowpass filter.

    Args:
        keypoints_2d: array containing the input data to be filtered.
        order: order of the butterworth filter. Larger numbers mean steeper response curves.
        lowcut: cutoff value above which frequencies are attenuated
        freq: sampling frequency of the input (60 Hz for stereo)

    Returns: 
        keypoints_2d_filtered: filtered version of the input 
    """
    # container to store results
    keypoints_2d_filtered = []

    # iterate by keypoints
    for keypoint in keypoints_2d.transpose(1, 0, 2):
        filtered_2d_keypoint = filter_2d_keypoint(
            keypoint=keypoint, order=order, lowcut=lowcut, fr=freq, create_diagram=False
        )
        keypoints_2d_filtered.append(filtered_2d_keypoint)

    # transform shape back to original
    keypoints_2d_filtered = np.array(keypoints_2d_filtered).transpose(1, 0, 2)
    return keypoints_2d_filtered


def get_3d_keypoints(keypoints_2d: np.ndarray, svo_file_path: str):
    """
    Get the depth value for each 2D keypoint in the video.

    Args: 
        keypoints_2d: array containing the 2D keypoints which are used to address the depth map created by stereo camera. (frames, num_kpts, dims)
        svo_file_path: path to the .svo2 file that was resorded by the stereo camera.

    Returns: 
        keypoints_3d: 3D keypoints corresponding to the 2D input. (frames, num_keypoints, (x,y,depth))
        Note, only the depth coordinate is in meters, the rest are still pixel values.
    """
    # results container
    keypoints_3d = []

    # create & configure camrea object
    zed = sl.Camera()
    input_type = sl.InputType()
    input_type.set_from_svo_file(svo_file_path)  # Set init parameter to run from the .svo
    init_parameters = sl.InitParameters(input_t=input_type, svo_real_time_mode=False)
    init_parameters.depth_mode = sl.DEPTH_MODE.NEURAL_PLUS
    init_parameters.coordinate_units = sl.UNIT.METER  # CENTIMETER, METER, MILLIMETER
    init_parameters.depth_stabilization = (
        75  # 0 to trun it off, otherwise 1-100 linear. default is 30
    )

    # Open the ZED
    err = zed.open(init_parameters)
    svo_depth = sl.Mat()
    svo_image = sl.Mat()

    frame_idx = 0

    while frame_idx < keypoints_2d.shape[0] - 1 and zed.grab() == sl.ERROR_CODE.SUCCESS:

        current_keypoints = []
        frame_idx = zed.get_svo_position()

        # get color image and depth map
        zed.retrieve_measure(svo_depth, sl.MEASURE.DEPTH)
        zed.retrieve_image(svo_image, sl.VIEW.LEFT)  # BGRA image

        img = svo_image.get_data()
        depth_map = svo_depth.get_data()
        depth_map = np.transpose(depth_map)

        # get coords for all keypoints within the current frame
        # note: x,y coords are in pixels, depth is in meters
        for x, y in keypoints_2d[frame_idx, :]:
            depth = depth_map[int(x), int(y)]
            current_keypoints.append(np.array([x, y, depth]))
            cv2.circle(img, (int(x), int(y)), 5, (0, 0, 255), 3)

        current_keypoints = np.array(current_keypoints)
        keypoints_3d.append(current_keypoints)
        cv2.imshow("vid", cv2.resize(img, (800, 600)))

        if cv2.waitKey(25) & 0xFF == ord("q"):
            break

    cv2.destroyAllWindows()
    zed.close()

    keypoints_3d = np.array(keypoints_3d)
    print(f"keypoints 3d shape: {keypoints_3d.shape} (frames, kpts, (x,y,depth))")

    return keypoints_3d


def get_extrinsics(
    calibration_img_path: str, camera_serial_number: str, debug_fig_path: str = None
):
    """
    Get the extrinsic camera parameters using a calibration image and the intrinsic camera parameters. 

    Args: 
        calibration_img_path: path of the calibration image
        camera_serial_number: serial number of the camera, e.g.: SN123456. Used to get the intrinsic parameters for the exact model.
        degub_fig_path: path to save a degub img to 

    Returns: 
        R: rotation matrix
        t_vec: translation vector
    """
    from Util.calibration_util import Pattern, get_image_points

    with open("./Util/camera_params.json", "r") as f:
        intrinsic_params = json.load(f)
        cameraMatrix = np.array(
            intrinsic_params[camera_serial_number]["left_sensor"]["camera_matrix"]
        )
        distCoeffs = np.array(
            intrinsic_params[camera_serial_number]["left_sensor"]["distortion_coeff"]
        )
        f_x = cameraMatrix[0, 0]
        f_y = cameraMatrix[1, 1]
        c_x = cameraMatrix[0, 2]
        c_y = cameraMatrix[1, 2]

    calibration_image = cv2.imread(calibration_img_path, cv2.IMREAD_GRAYSCALE)

    # describe real world calibration pattern parameteres
    board_pattern = Pattern(height=6, width=4, square_size=0.150)

    camera_coords = []

    # get specific points of the image used for calibration, associate them with real-life points of calib pattern
    img_points, obj_points, debug_img = get_image_points(calibration_image, board_pattern)

    # get the transformations describing image-object relationship
    success, rvec, t_vec = cv2.solvePnP(
        obj_points,
        img_points,
        cameraMatrix,
        distCoeffs,
        useExtrinsicGuess=False,
        flags=cv2.SOLVEPNP_ITERATIVE,
    )

    if success:
        R, jac = cv2.Rodrigues(rvec)

        Xw = -np.matrix(R).T * np.matrix(t_vec)
        camera_coords.append(Xw)

        print(f"distance: {round(np.linalg.norm(Xw),2)} m")

    else:
        print(f"{fn}... FAILED")

    # TODO: finish debug img saving
    # if debug_fig_path is not None:
    #     out_file_name =
    #     cv2.imwrite()
    return R, t_vec


def transform_to_global_frame(
    camera_frame_keypoints: np.ndarray, rotation_matrix: np.ndarray, translation_vector: np.ndarray
):
    """
    Transforms 3D keypoints from camera-frame to global-frame using extrinsic parameters obtained by calibration. 
    
    Args:
        camera_frame_keypoints: 3D keypoints represented in camrea based coordinate system
        rotation_matrix: R, currently not used.
        translation_vector: describes translation relative to calibration pattern
    
    Returns:
        keypoints_3d_global_frame: 3D keypoints represented in the global coordinate system. (frames, kpts, (depth, horizontal, vertical))
    
    """
    # TODO: get rid of loading camera params twice
    with open("./Util/camera_params.json", "r") as f:
        intrinsic_params = json.load(f)
        # TODO: replace hardcoded camera serial number
        cameraMatrix = np.array(intrinsic_params["SN43916681"]["left_sensor"]["camera_matrix"])
        f_x = cameraMatrix[0, 0]
        f_y = cameraMatrix[1, 1]
        c_x = cameraMatrix[0, 2]
        c_y = cameraMatrix[1, 2]

    keypoints_3d_global_frame = []

    for kpt in camera_frame_keypoints.transpose(1, 2, 0):
        # camera-frame coordinates
        u = kpt[0, :]
        v = kpt[1, :]
        Zc = kpt[2, :]

        # global/world-frame coordinates
        Xw = ((u - c_x) * Zc / f_x) - translation_vector[0]
        Yw = ((v - c_y) * Zc / f_y) - translation_vector[1]
        Zw = Zc - translation_vector[2]

        # multiply by -1 so axes go in the correct direction
        Yw = -Yw
        Zw = -Zw

        # (depth, horizontal, vertical)
        keypoints_3d_global_frame.append(np.array([Zw, Xw, Yw]))

    keypoints_3d_global_frame = np.array(keypoints_3d_global_frame).transpose(2, 0, 1)

    return keypoints_3d_global_frame


def resample_timeseries(keypoints_3d: np.ndarray, original_fps: float, target_fps: float):
    """
    Resample the input time series data to the desired frequency.

    Args:
        keypoints_3d: 3D keypoints in some coord system
        original_fps: original sampling frequency in Hz of time series
        target_fps: desired sampling frequency

    Returns: 
        resampled_keypoints: keypoint data resampled with the desired frequency
    """
    original_num_frames = keypoints_3d.shape[0]
    target_num_frames = int(np.round(original_num_frames * target_fps / original_fps))
    resampled_keypoints = resample(keypoints_3d, target_num_frames, axis=0, window=100)
    return resampled_keypoints


def visualize_2d_keypoints(keypoints_2d: np.ndarray, video_path: str) -> None:
    """
    Draw 2D keypoint locations on each frame of RGB video.

    Args:
        keypoints_2d: 2D keypoints to be drawn
        video_path: path to video used to obtain keypoints

    Returns:
        None
    """
    # labels are loaded from static file, only works for stereo 
    labels = models["pose_models"]["26"]["kpt_labels"]

    cap = cv2.VideoCapture(video_path)
    if cap.isOpened() == False:
        print("Error opening video file")

    frame_idx = 0

    # Read until video is completed
    while cap.isOpened() and frame_idx < keypoints_2d.shape[0] - 1:

        # Capture frame-by-frame
        ret, frame = cap.read()

        if ret == True and frame is not None:
            frame_idx += 1
            h, w = frame.shape[:2]

            for kpt_idx, kpt in enumerate(keypoints_2d[frame_idx, :, :]):
                # highlight certain keypoints in green
                if kpt_idx in [
                    labels.index("left_ankle"),
                    labels.index("right_ankle"),
                ]:
                    color = (0, 255, 0)
                else:
                    color = (0, 0, 255)
                cv2.circle(frame, (int(kpt[0]), int(kpt[1])), radius=3, color=color, thickness=2)
            cv2.imshow("Image", cv2.resize(frame, (int(w / 1.5), int(h / 1.5))))
            time.sleep(0.01)

            # Press Q on keyboard to exit
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break
        else:
            break
    cap.release()
    cv2.destroyAllWindows()

In [7]:
# GET 2D KEYPOINTS
stereo_labels = models['pose_models']["26"]["kpt_labels"]
video_file_path = f"stereo_videos\\validation_test\\{participant}\\{paradigm}\\43916681.avi"
# video_file_path = "./20260415_134043.mp4"

keypoints_2d = get_2d_keypoints(video_file_path=video_file_path, labels=stereo_labels)

# save raw 2d keypoints
with open('./filter_test_2d.npz', "w"):
    np.savez(
        './filter_test_2d.npz',
        keypoints_2d=keypoints_2d,
        labels=stereo_labels
    )

shape of keypoints_2d:  (3908, 27, 2) (frames * kpt * (x, y))


In [ ]:
# FILTER 2D
keypoints_2d_filtered_euro = filter_2d_keypoints_euro(keypoints_2d=keypoints_2d,freq=60,min_cutoff=2,beta=0.1)
keypoints_2d_filtered_butter = filter_2d_keypoints_butter(keypoints_2d=keypoints_2d,order=4,lowcut=2,freq=60)

In [ ]:
# VISUALIZATION
import time

input_vid = f".\\stereo_videos\\validation_test\\{participant}\\{paradigm}\\43916681.avi"
visualize_2d_keypoints(keypoints_2d=keypoints_2d, video_path=input_vid)

In [ ]:
# GET 3D KEYPOINTS
svo_file_path = f"stereo_videos\\validation_test\\{participant}\\{paradigm}\\43916681.svo2"
keypoints_3d_euro = get_3d_keypoints(keypoints_2d=keypoints_2d_filtered_euro, svo_file_path=svo_file_path)
keypoints_3d_butter = get_3d_keypoints(keypoints_2d=keypoints_2d_filtered_butter, svo_file_path=svo_file_path)

In [ ]:
# SAVE RAW 3D KEYPOINTS
with open('./filter_test_3d.npz', "w"):
    np.savez(
        './filter_test_3d.npz',
        keypoints_2d=keypoints_2d,
        labels=stereo_labels,
        keypoints_3d_euro=keypoints_3d_euro,
        keypoints_3d_butter=keypoints_3d_butter
    )

In [ ]:
# LOAD RAW 3D KEYPOINTS
loaded_data = np.load('./filter_test_3d.npz')
keypoints_2d = loaded_data['keypoints_2d']
keypoints_3d_euro = loaded_data['keypoints_3d_euro']
keypoints_3d_butter = loaded_data['keypoints_3d_butter']
stereo_labels = loaded_data['labels'].tolist()

In [ ]:
# TRANSFORM TO GLOBAL COORDS
calibraiton_img_path = f"stereo_videos\\validation_test\\{participant}\\{paradigm}\\stereo_calibration.png"
R, t = get_extrinsics(calibraiton_img_path, camera_serial_number='SN43916681', debug_fig_path=None)
keypoints_3d_euro = transform_to_global_frame(camera_frame_keypoints=keypoints_3d_euro,rotation_matrix=R, translation_vector=t)
keypoints_3d_butter = transform_to_global_frame(camera_frame_keypoints=keypoints_3d_butter, rotation_matrix=R, translation_vector=t)

In [ ]:
# FILTER 3D KEYPOINTS
keypoints_3d_euro = filter_data(data=keypoints_3d_euro.transpose(1, 2, 0),sampling_rate=60,filter_type="lowpass",cutoff=[1.5],order=4,gap_size=0.25,).transpose(2,0,1)
keypoints_3d_butter = filter_data(data=keypoints_3d_butter.transpose(1, 2, 0),sampling_rate=60,filter_type="lowpass",cutoff=[1.5],order=4,gap_size=0.25,).transpose(2,0,1)

In [ ]:
# RESAMPLE TO 100 Hz
keypoints_3d_euro = resample_timeseries(keypoints_3d=keypoints_3d_euro,original_fps=60,target_fps=100)
keypoints_3d_butter = resample_timeseries(keypoints_3d=keypoints_3d_butter,original_fps=60,target_fps=100)

In [ ]:
# POSITION PLOT

qualisys_keypoints = qualisys_keypoints.transpose(2,0,1)

r_ankle_idx = stereo_labels.index("right_ankle")
l_ankle_idx = stereo_labels.index("left_ankle")
r_ankle_idx_q = qualisys_labels.index("right_ankle")

fig, axs = plt.subplots(3,1, figsize=(14,4))
axs[0].plot(keypoints_3d_butter[:,r_ankle_idx,0], label="butter")
axs[0].plot(keypoints_3d_euro[:,r_ankle_idx,0], label="euro")
axs[0].plot(qualisys_keypoints[:,r_ankle_idx_q,0], label="qualisys")

axs[1].plot(keypoints_3d_butter[:,r_ankle_idx,1], label="butter")
axs[1].plot(keypoints_3d_euro[:,r_ankle_idx,1], label="euro")
axs[1].plot(qualisys_keypoints[:,r_ankle_idx_q,1], label="qualisys")

axs[2].plot(keypoints_3d_butter[:,r_ankle_idx,2], label="butter")
axs[2].plot(keypoints_3d_euro[:,r_ankle_idx,2], label="euro")
axs[2].plot(qualisys_keypoints[:,r_ankle_idx_q,2], label="qualisys")

for ax in axs:
    ax.legend()
plt.tight_layout()

In [ ]:
# VELOCITY PLOT (EACH AXIS)
fig, axs = plt.subplots(3,1, figsize=(14,6))
axs[0].plot(np.diff(keypoints_3d_butter[:,r_ankle_idx,0]), label="butter")
axs[0].plot(np.diff(keypoints_3d_euro[:,r_ankle_idx,0]), label="euro")
axs[0].plot(np.diff(qualisys_keypoints[:,r_ankle_idx_q,0]), label="qualisys")

axs[1].plot(np.diff(keypoints_3d_butter[:,r_ankle_idx,1]), label="butter")
axs[1].plot(np.diff(keypoints_3d_euro[:,r_ankle_idx,1]), label="euro")
axs[1].plot(np.diff(qualisys_keypoints[:,r_ankle_idx_q,1]), label="qualisys")

axs[2].plot(np.diff(keypoints_3d_butter[:,r_ankle_idx,2]), label="butter")
axs[2].plot(np.diff(keypoints_3d_euro[:,r_ankle_idx,2]), label="euro")
axs[2].plot(np.diff(qualisys_keypoints[:,r_ankle_idx_q,2]), label="qualisys")


axs[0].set_title("depth")
axs[1].set_title("x")
axs[2].set_title("y")
for ax in axs:
    ax.legend()
plt.tight_layout()

In [ ]:
# 3D VELOCITY PLOT
fig, axs = plt.subplots(2,1, figsize=(14,4))

shift=400

axs[0].plot(np.mean(keypoints_3d_euro[shift:,r_ankle_idx,:],axis=1),label="euro")
axs[0].plot(np.mean(keypoints_3d_butter[shift:,r_ankle_idx,:],axis=1),label="butter")
axs[0].plot(np.mean(qualisys_keypoints[:,r_ankle_idx_q,:],axis=1),label="q")

axs[1].plot(np.diff(np.mean(keypoints_3d_euro[shift:,r_ankle_idx,:],axis=1)),label="euro")
axs[1].plot(np.diff(np.mean(keypoints_3d_butter[shift:,r_ankle_idx,:],axis=1)),label="butter")
axs[1].plot(np.diff(np.mean(qualisys_keypoints[:,r_ankle_idx_q,:],axis=1))[:-200],label="q")


axs[0].set_title("3D position of 1 kpt")
axs[1].set_title("3D velocity of 1 kpt")
for ax in axs:
    ax.legend()
plt.tight_layout()

In [ ]:
# SAVE FILTERED 3D KEYPOINTS
stereo_save_path_euro = f".\\filter_test_data\\stereo_euro.npz"
stereo_save_path_butter = f".\\filter_test_data\\stereo_butter.npz"

qualisys_save_path = f".\\filter_test_data\\qualisys.npz"
debug_folder_path = f".\\filter_test_data"

with open(stereo_save_path_euro, "w"):
    np.savez(
        stereo_save_path_euro,
        labels=stereo_labels,
        keypoints=keypoints_3d_euro.transpose(1,2,0)
    )   
with open(stereo_save_path_butter, "w"):
    np.savez(
        stereo_save_path_butter,
        labels=stereo_labels,
        keypoints=keypoints_3d_butter.transpose(1,2,0)
    )   
with open(qualisys_save_path, "w"):
    np.savez(
        qualisys_save_path,
        labels=stereo_labels,
        keypoints=qualisys_keypoints.transpose(1,2,0)
    ) 

In [ ]:
# LOAD FILTERED 3D KEYPOITNS QUALISYS

qualisys_tsv_path = ".\\filter_test_data\\A2A6841F-E163-433C-BEF4-55BD9C15437A.tsv"
format_qualisys_export(qualisys_tsv_path)
qualisys_npz_path = f".\\filter_test_data\\qualisys.npz"
qualisys_keypoints, qualisys_labels = load_keypoints_and_labels(qualisys_npz_path)
qualisys_keypoints = filter_data(data=qualisys_keypoints,sampling_rate=100,filter_type="lowpass",cutoff=[7],order=4, gap_size=0.25)

In [ ]:
with open(qualisys_npz_path, "w"):
    np.savez(
        qualisys_npz_path,
        labels=qualisys_labels,
        keypoints=qualisys_keypoints
    ) 

In [ ]:
# LOAD FILTERED 3D KEYPOINTS
stereo_save_path_euro = f".\\filter_test_data\\stereo_euro.npz"
stereo_save_path_butter = f".\\filter_test_data\\stereo_butter.npz"
qualisys_save_path = f".\\filter_test_data\\qualisys.npz"
debug_folder_path = f".\\filter_test_data"

stereo_keypoints_euro, stereo_labels_euro = load_keypoints_and_labels(stereo_save_path_euro)
stereo_keypoints_butter, stereo_labels_butter = load_keypoints_and_labels(stereo_save_path_butter)
qualisys_keypoints, qualisys_labels = load_keypoints_and_labels(qualisys_save_path)

In [ ]:
recording_euro = Recording(
    keypoints=stereo_keypoints_euro,
    labels=stereo_labels_euro,
    recording_type="stereo",
    gait_analysis_properties=gait_analysis_properties,
    debug_folder=debug_folder_path,
)

In [ ]:
recording_butter = Recording(
    keypoints=keypoints_3d_butter.transpose(1,2,0),
    labels=stereo_labels,
    recording_type="stereo",
    gait_analysis_properties=gait_analysis_properties,
    debug_folder=debug_folder_path,
)

In [ ]:
recording_qualisys = Recording(
    keypoints=qualisys_keypoints.transpose(1,2,0),
    labels=qualisys_labels,
    recording_type="qualisys",
    gait_analysis_properties=gait_analysis_properties,
    debug_folder=debug_folder_path,
)

In [ ]:
fig= plt.figure(figsize=(14,2))

gait_events = recording_euro.gait_events

for ge in gait_events:
    alpha = 1 if ge.segment=='straight' else 0.2
    color = "#55e05c" if ge.type=='IC' else "#5565e0"
    match ge.side:
        case 'left':
            plt.vlines(x=ge.frame, ymin=0, ymax=1, alpha=alpha, colors=color)
        case 'right':
            plt.vlines(x=ge.frame, ymin=-1, ymax=0, alpha=alpha, colors=color)
plt.legend()
plt.tight_layout()

## GE detection rework

In [ ]:
velocity = 0.5
fps = 100
side = "left"
stance_time_min = 0.3 * fps
swing_time_min = 0.3 * fps

# keypoints = stereo_keypoints_butter
# labels = stereo_labels_butter
keypoints = qualisys_keypoints
labels = qualisys_labels

turn_mask, perspective = get_turns_and_perspective(
    keypoint_data=keypoints, kpt_labels=labels, sampling_fr=100, debug_fig_file_path="./"
)
# turn_mask = np.where(turn_mask == 1, "turn", "straight")

heel_thr = 0.5 * velocity
ankle_thr = 0.5 * velocity
big_toe_thr = 0.8 * velocity

all_events = []

heel = keypoints[labels.index(f"{side}_heel"), :, :]
big_toe = keypoints[labels.index(f"{side}_big_toe"), :, :]
ankle = keypoints[labels.index(f"{side}_ankle"), :, :]
ankle_q = qualisys_keypoints[qualisys_labels.index(f"{side}_ankle")]

heel_vel = np.linalg.norm(np.diff(heel, axis=1, prepend=heel[:, [0]]), axis=0) * fps
ankle_vel = np.linalg.norm(np.diff(ankle, axis=1, prepend=ankle[:, [0]]), axis=0) * fps
big_toe_vel = np.linalg.norm(np.diff(big_toe, axis=1, prepend=big_toe[:, [0]]), axis=0) * fps
ankle_q_vel = np.linalg.norm(np.diff(ankle_q, axis=1, prepend=ankle_q[:, [0]]), axis=0) * fps

ground_contact_thrs = (ankle_vel < ankle_thr).astype(int)

ground_contact_straight = np.where(turn_mask == False, ground_contact_thrs, 1)

# get preliminary FC & IC events based on ground contact
contact_diff_straight = np.diff(ground_contact_straight,prepend=ground_contact_straight[0])

# remove too short swing periods
ground_conact_no_short_swing = np.copy(ground_contact_straight)
starts = np.nonzero(contact_diff_straight == -1)[0]
ends = np.nonzero(contact_diff_straight == 1)[0]

for start_frame,end_frame in zip(starts,ends):
    if end_frame-start_frame < swing_time_min:
        ground_conact_no_short_swing[start_frame:end_frame] = 1

contact_diff_no_short_swing = np.diff(ground_conact_no_short_swing,prepend=ground_conact_no_short_swing[0])

# remove too short contact periods
ground_contact_no_short_contact = np.copy(ground_conact_no_short_swing)
starts = np.nonzero(contact_diff_no_short_swing == 1)[0]
ends = np.nonzero(contact_diff_no_short_swing == -1)[0]
for idx, [start_frame, end_frame] in enumerate(zip(starts, ends)):
    if idx+1 < ends.shape[0]:
        if ends[idx+1] - start_frame < stance_time_min:
            ground_contact_no_short_contact[start_frame:ends[idx+1]] = 0

contact_diff_no_short_contact = np.diff(ground_contact_no_short_contact, prepend=ground_contact_no_short_contact[0])


In [ ]:
plt.close('all')
fig, axs = plt.subplots(4,1,figsize=(18,7))
axs[0].plot(np.pad(array=ankle_q_vel,pad_width=(450,0)))

axs[1].plot(ankle_vel)
axs[1].plot(ground_contact_thrs*(ankle_thr*velocity))
axs[1].plot(turn_mask*5,'r--', alpha=0.45)

axs[2].plot(ground_contact_thrs+6, label="raw", color="#ff8800ff")
axs[2].plot(ground_contact_straight+4, label="no turns", color="#ffd000ff")
axs[2].plot(ground_conact_no_short_swing+2, label="no short swings",color="#d9e717")
axs[2].plot(ground_contact_no_short_contact, label="no short contacts",color="#6dc916")

axs[3].plot(contact_diff_straight+4, label="no turns", color="#ffd000ff")
axs[3].plot(contact_diff_no_short_swing+2, label="no short swings",color="#d9e717")
axs[3].plot(contact_diff_no_short_contact, label="no short contact",color="#6dc916")

# format
axs[0].set_title("qualisys_ankle")
axs[1].set_title("stereo_ankle")
axs[2].set_title("ground contact")
axs[3].set_title("ground contact diff (GEs)")
axs[2].legend()
axs[3].legend()
for ax in axs[:2]:
    ax.set_ylim([0,3.5])

# for ax in axs:
#     ax.set_xlim([1000,7000])
  
plt.tight_layout()

In [ ]:
plt.close('all')
fig, axs = plt.subplots(4,1,figsize=(18,7))
axs[0].plot(ankle_vel)
axs[1].plot(heel_vel)
axs[2].plot(big_toe_vel)
axs[3].plot((ankle_vel+heel_vel+big_toe_vel)/3)

for ax in axs:
    ax.set_ylim([0,3.2])

In [ ]:


# # remove too short swing periods
# contact_diff2 = np.diff(np.pad(ground_contact, 1, "constant",constant_values=0))
# starts = np.nonzero(contact_diff2 == 1)[0]
# ends = np.nonzero(contact_diff2 == -1)[0]
# for start, end in zip(starts, ends):
#     if end - start < min_swing_time:
#         ground_contact[start:end] = 1

# contact_diff3 = np.diff(np.pad(ground_contact, 1, "constant",constant_values=0))

# IC_frames = np.nonzero(contact_diff3 == 1)[0] + 1
# FC_frames = np.nonzero(contact_diff3 == -1)[0] + 1

# IC_events = [
#     GaitEvent(
#         type="IC",
#         frame=frame,
#         side=side,
#         perspective=perspective[frame],
#         segment=turn_mask[frame],
#     )
#     for frame in IC_frames if frame >  100
# ]

# FC_events = [
#     GaitEvent(
#         type="FC",
#         frame=frame,
#         side=side,
#         perspective=perspective[frame],
#         segment=turn_mask[frame],
#     )
#     for frame in FC_frames if frame < len(turn_mask)
# ]

# all_events.extend(IC_events + FC_events)

# # PLOT
# if True:
#     t = np.linspace(0, len(heel_vel) / fps, len(heel_vel))
#     plt.close("all")
#     fig, axs = plt.subplots(5, 1, figsize=(14, 9))

#     axs[0].plot(ankle_vel, label="velocity")
#     axs[0].plot(ground_contact * ankle_thr, label="ground contact at thr")

#     axs[1].plot(big_toe_vel, label="velocity")
#     axs[1].plot(ground_contact * big_toe_thr, label="ground contact at thr")

#     axs[2].plot(heel_vel, label="velocity")
#     axs[2].plot(ground_contact * heel_thr, label="ground contact at thr")

#     axs[3].plot(contact_diff3, alpha=0.6, label="ground contact diff")
#     axs[3].plot(turn_mask, "r--", label="turn mask")

#     axs[4].plot(perspective[1:], label="perspective")

#     # fmt: off
#     axs[0].set_title("ankle velocity")
#     axs[1].set_title("toe velocity")
#     axs[2].set_title("heel velocity")
#     axs[3].set_title("ground contact diff (gait events)")
#     axs[0].set_yticks([0, 3, ankle_thr])
#     axs[0].set_ylim([0,3])
#     axs[1].set_yticks([0, 3, big_toe_thr])
#     axs[1].set_ylim([0,3])
#     axs[2].set_yticks([0, 3, heel_thr])
#     axs[2].set_ylim([0,3])
#     axs[3].set_yticks([-1,1,],["FC", "IC"],)
#     axs[3].set_yticks([-1,1,],["FC", "IC"],)
#     axs[4].set_yticks([0, 1], ["back", "front"])
#     # fmt: on
#     for ax in axs:
#         ax.legend(loc="upper left")
#         ax.grid()

#     plt.tight_layout()


#     plt.close("all")
#     print(f"GE figure saved to: \n{out_filename}")
# all_events_sorted = sorted(all_events, key=lambda event: event.frame)

## recording synchronization

### using first step event

In [ ]:
stereo_first_event = Pair.stereo.gait_events[0].frame
qualisys_first_event = Pair.qualisys.gait_events[0].frame
print(f"stereo first: {stereo_first_event}")
print(f"qualisys first: {qualisys_first_event}")

In [ ]:
abs(stereo_first_event - qualisys_first_event)

In [ ]:
q_ics = [gc.IC0.frame for gc in Pair.qualisys.gait_cycles if gc.flags.valid]
q_ics_vis = np.zeros(Pair.qualisys.keypoints.shape[2])
q_ics_vis[q_ics] = 1

s_ics = [gc.IC0.frame for gc in Pair.stereo.gait_cycles if gc.flags.valid]
s_ics_vis = np.zeros(Pair.stereo.keypoints.shape[2])
s_ics_vis[s_ics] = 1

shift = abs(Pair.stereo.gait_events[0].frame - Pair.qualisys.gait_events[0].frame)

plt.close('all')
fig, axs = plt.subplots(2,1, figsize=(14,3))

axs[0].plot(s_ics_vis[shift:], color='#1f77b4')
axs[0].plot(Pair.stereo.turn_mask[shift:],color='#ff7f0e', alpha=0.5)

axs[1].plot(q_ics_vis,color='#1f77b4')
axs[1].plot(Pair.qualisys.turn_mask,color='#ff7f0e',alpha=0.5)

# axs[2].plot(s_ics_vis,color='#1f77b4')
# axs[2].plot(Pair.stereo.turn_mask,color='#ff7f0e',alpha=0.5)

# axs[3].plot(q_ics_vis,color='#1f77b4')
# axs[3].plot(Pair.qualisys.turn_mask,color='#ff7f0e',alpha=0.5)

for stereo_gc, qualisys_gc in pairs:
    stereo_frame = stereo_gc.IC0.frame
    qualisys_frame = qualisys_gc.IC0.frame

    axs[0].plot(stereo_frame-shift,0.5, marker='o')
    axs[1].plot(qualisys_frame, 0.5, marker='o')

axs[0].set_title('stereo')
axs[1].set_title('qualisys')
# axs[2].set_title('original stereo')
# axs[3].set_title('original qualisys')

axs[0].set_xlim([0,6100])
axs[1].set_xlim([0,6100])

plt.tight_layout()
plt.show()

In [ ]:
pairs = []
for idx, gc in enumerate(Pair.stereo.gait_cycles):
    if gc.flags.valid:
        sync_frame = gc.IC0.frame - shift

        potential_pairs = [
            p
            for p in Pair.qualisys.gait_cycles
            if abs(p.IC0.frame - sync_frame) < 60
            and p.flags.valid
            and p.ipsi == gc.ipsi
            and p.perspective == gc.perspective
            and p.IC0.frame + shift + 10 >= gc.IC0.frame
        ]
        if len(potential_pairs) != 0:
            pp_frames = [c.IC0.frame for c in potential_pairs]
            pair = potential_pairs
            pairs.append([gc, potential_pairs[0]])

            print(sync_frame, gc)
            print(potential_pairs[0].IC0.frame, potential_pairs[0])
            print(idx, pp_frames)
            print("")

In [ ]:
for stereo_event, qualisys_event in zip(Pair.stereo.gait_events, Pair.qualisys.gait_events):
    print(f"({stereo_event.frame}) {stereo_event.frame-shift}  |  {qualisys_event.frame},   d: {stereo_event.frame-shift-qualisys_event.frame}")

### using perspective

In [ ]:
stereo_starts = np.nonzero(sp_diff == 1)[0]
stereo_ends = np.nonzero(sp_diff == -1)[0]
print(len(stereo_starts), len(stereo_ends))
print(stereo_starts)
print(stereo_ends)
for start, end in zip(stereo_starts[:-1], stereo_ends[:-1]):
    print(round((end-start)/Pair.stereo.fps,3))

In [ ]:
qualisys_starts = np.nonzero(qp_diff == 1)[0]
qualisys_ends = np.nonzero(qp_diff == -1)[0]
print(len(qualisys_starts), len(qualisys_ends))
print(qualisys_starts)
print(qualisys_ends)
for start, end in zip(qualisys_starts[:-1], qualisys_ends[:-1]):
    print(round((end-start)/Pair.qualisys.fps,3))

In [ ]:
stereo_first = stereo_starts[0]
stereo_last = stereo_ends[-2]
qualisys_first = qualisys_starts[0]
qualisys_last = qualisys_ends[-2]

start_diff_sec = stereo_first/Pair.stereo.fps - qualisys_first/Pair.qualisys.fps
start_diff = int(round(start_diff_sec * Pair.stereo.fps))

end_diff_sec = stereo_last/Pair.stereo.fps - qualisys_last/Pair.qualisys.fps
end_diff = int(round(end_diff_sec * Pair.qualisys.fps))

print(f"start_diff: {round(start_diff_sec,2)} s, {round(start_diff,2)} samples (stereo fps)")
print(f"start_diff: {round(end_diff_sec,2)} s, {round(end_diff,2)} samples (qualisys fps)")

In [ ]:
qp = np.where(Pair.qualisys.perspective=='back',1,0)
sp = np.where(Pair.stereo.perspective=='back',1,0)
qp_diff = np.diff(np.pad(qp,1,mode="constant"))
sp_diff = np.diff(np.pad(sp,1,mode="constant"))


fig, axs = plt.subplots(4,1,figsize=(14,5))
axs[0].plot(sp)
axs[0].plot(sp_diff)

axs[1].plot(qp)
axs[1].plot(qp_diff)

axs[2].plot(sp[start_diff:])
axs[3].plot(qp[:-130])

plt.tight_layout()

In [ ]:
q_ics = [ge.frame for ge in Pair.qualisys.gait_events if ge.type=='IC']
q_ics_vis = np.zeros(Pair.qualisys.keypoints.shape[2])
q_ics_vis[q_ics] = 1

s_ics = [ge.frame for ge in Pair.stereo.gait_events if ge.type=='IC']
s_ics_vis = np.zeros(Pair.stereo.keypoints.shape[2])
s_ics_vis[s_ics] = 1


plt.close('all')
fig, axs = plt.subplots(4,1, figsize=(14,5))

axs[0].plot(s_ics_vis)
axs[0].plot(Pair.stereo.perspective)
axs[1].plot(q_ics_vis)
axs[1].plot(Pair.qualisys.perspective)

axs[2].plot(s_ics_vis)
axs[2].plot(Pair.stereo.perspective)
axs[3].plot(q_ics_vis)
axs[3].plot(Pair.qualisys.perspective)

axs[0].set_title('stereo')
axs[1].set_title('qualisys')
axs[2].set_title('original stereo')
axs[3].set_title('original qualisys')


plt.tight_layout()
plt.show()

### using metadata files

In [ ]:
stereo_len = round(Pair.stereo.keypoints.shape[2] / Pair.stereo.fps,2) # seconds
qualisys_len = round(Pair.qualisys.keypoints.shape[2] / Pair.qualisys.fps,2) # seconds

stereo_shift = get_synchronisation_shift(qualisys_start_file_path)
new_stereo_len = round(stereo_len-stereo_shift,2)
diff = qualisys_len-new_stereo_len

print(f"og stereo length: {stereo_len:>10}")
print(f"og qualisys length: {qualisys_len:>8}")
print(f"new stereo len: {new_stereo_len:>12}")
print(f"diff :{diff:>26}")

In [ ]:
start_shift = get_synchronisation_shift(qualisys_start_file_path)
stop_shift = get_synchronisation_shift(qualisys_stop_file_path)

print(f"start: {start_shift} s")
print(f"stop:  {stop_shift} s")
start_shift-stop_shift

In [ ]:
(Pair.stereo.keypoints.shape[2]-stereo_shift)/60 *100, Pair.qualisys.keypoints.shape


In [ ]:
q_ics = [ge.frame for ge in Pair.qualisys.gait_events if ge.type=='IC']
q_ics_vis = np.zeros(Pair.qualisys.keypoints.shape[2])
q_ics_vis[q_ics] = 1

s_ics = [ge.frame for ge in Pair.stereo.gait_events if ge.type=='IC']
s_ics_vis = np.zeros(Pair.stereo.keypoints.shape[2])
s_ics_vis[s_ics] = 1

qualisys_fps = Pair.qualisys.fps
stereo_fps = Pair.stereo.fps
stereo_shift = int(get_synchronisation_shift(qualisys_start_file_path)*100)
# qualisys_shift = Pair.qualisys.keypoints.shape[2] - int(get_synchronisation_shift(qualisys_stop_file_path)*gait_analysis_properties['qualisys']['fps'])
# qualisys_shift = Pair.qualisys.keypoints.shape[2] - int(diff *gait_analysis_properties['qualisys']['fps'])
q_shift = int((Pair.stereo.keypoints.shape[2]-stereo_shift)/60 *100)

plt.close('all')
fig, axs = plt.subplots(4,1, figsize=(14,5))

axs[0].plot(s_ics_vis[stereo_shift:])
axs[0].plot(Pair.stereo.turn_mask[stereo_shift:])
axs[1].plot(q_ics_vis)
axs[1].plot(Pair.qualisys.turn_mask)

axs[2].plot(s_ics_vis)
axs[2].plot(Pair.stereo.turn_mask)
axs[3].plot(q_ics_vis)
axs[3].plot(Pair.qualisys.turn_mask)

axs[0].set_title('stereo')
axs[1].set_title('qualisys')
axs[2].set_title('original stereo')
axs[3].set_title('original qualisys')

axs[0].set_xlim([0,6100])
axs[1].set_xlim([0,6100])

plt.tight_layout()
plt.show()

In [ ]:
from scipy.signal import resample

stereo_keypoints = Pair.stereo.keypoints  # shape (n_kpts, 3, n_frames)
orig_fps = Pair.stereo.fps  # 60
target_fps = 100

n_frames = stereo_keypoints.shape[2]
n_frames_new = int(np.round(n_frames * target_fps / orig_fps))

stereo_keypoints_100fps = resample(stereo_keypoints, n_frames_new, axis=2)

# overwrite or keep a new variable
Pair.stereo.keypoints = stereo_keypoints_100fps
Pair.stereo.fps = target_fps

In [ ]:
Pair.qualisys.gait_events

In [ ]:
Pair.stereo.gait_events

In [ ]:
Pair.stereo.keypoints.shape

In [ ]:
t_nano = 1741700117299209216
t_sec = t_nano/1000000000

In [ ]:
from datetime import datetime

metadata_file_path = f"./stereo_videos\\validation_test\\{participant}\\{paradigm}\\data_start_qualisys.json"

with open(metadata_file_path,'r') as f:
    loaded_metadata = json.load(f)
    timestamp = datetime.fromisoformat(loaded_metadata['timestamp'])
    timestamp_qualisys_sec = int(loaded_metadata['timestamp_qualisys'])/1e9
    timestamp_qualisys = datetime.fromtimestamp(timestamp_qualisys_sec,tz=timestamp.tzinfo)
timestamp, timestamp_qualisys

In [ ]:
time_shift = (timestamp_qualisys-timestamp).total_seconds()
print(f"time_shift: {time_shift} seconds")

In [ ]:
# GE detection & false positive rejection plot
fig, axs = plt.subplots(4,1,figsize=(15,4))

'right' = 'left'

heel_thr = R.heel_thr * 1
ankle_thr = R.heel_thr * 1
big_toe_thr = R.toe_thr * 1

heel = R.keypoints[R.labels.index(f"{'right'}_heel"), :, :]
big_toe = R.keypoints[R.labels.index(f"{'right'}_big_toe"), :, :]
ankle = R.keypoints[R.labels.index(f"{'right'}_ankle"), :, :]

heel_vel = np.linalg.norm(np.diff(heel, axis=1), axis=0) * R.fps
ankle_vel = np.linalg.norm(np.diff(ankle, axis=1), axis=0) * R.fps
big_toe_vel = np.linalg.norm(np.diff(big_toe, axis=1), axis=0) * R.fps

ground_contact = (
    (heel_vel < heel_thr) | (ankle_vel < ankle_thr) | (big_toe_vel < big_toe_thr)
).astype(int)
axs[0].plot(ground_contact)
axs[0].plot(R.turn_mask)


# ground_contact[self.turn_mask] = 0
# ground_contact = np.where(self.turn_mask[1:]=='turn',ground_contact,0)

contact_diff1 = np.diff(
    np.pad(ground_contact, 1, "constant")
)  # NOTE:pad beginning and end of array with 0 for diff

# remove too short contact periods
starts = np.nonzero(contact_diff1 == 1)[0]
ends = np.nonzero(contact_diff1 == -1)[0]
for start, end in zip(starts, ends):
    if end - start < R.min_stance_time:
        ground_contact[start:end] = 0
axs[1].plot(ground_contact)

# remove too short swing periods
contact_diff2 = np.diff(np.pad(ground_contact, 1, "constant"))
starts = np.nonzero(contact_diff2 == 1)[0]
ends = np.nonzero(contact_diff2 == -1)[0]
for start, end in zip(starts, ends):
    if end - start < R.min_swing_time:
        ground_contact[start:end] = 1

axs[2].plot(ground_contact)
contact_diff3 = np.diff(np.pad(ground_contact, 1, "constant"))
ICs = np.nonzero(contact_diff3 == 1)[0] + 1
FCs = np.nonzero(contact_diff3 == -1)[0] + 1


# axs[1].plot(contact_diff1)
# axs[2].plot(contact_diff2)
# axs[3].plot(contact_diff3)
plt.tight_layout()

## Custom statistics

In [ ]:
method_A_data = np.array([0,1,2,3,4])
method_B_data = np.array([5,6,4,7,9])
print(method_A_data)
print(method_B_data)

In [ ]:
icc_results = icc_statistics(method_A_data,method_B_data)
icc_results

In [ ]:
ICC_3_1 = float(icc_results.loc[icc_results["Type"]=="ICC3"]["ICC"].iloc[0])
ICC_3_1

In [ ]:
icc_results

In [ ]:
subjects = list(range(1, len(method_A_data) + 1))

# Prepare the DataFrame

data = pd.DataFrame(
    {
        "Subject_ID": subjects + subjects,  # Repeat subject IDs for each method
        "Measurement": list(method_A_data) + list(method_B_data),  # Combine measurements
        "Method": ["A"] * len(method_A_data) + ["B"] * len(method_B_data),  # Label methods
    }
)

# Convert "Method" to categorical
data["Method"] = pd.Categorical(data["Method"])
data

## PLY

In [ ]:
import open3d as o3d

In [ ]:
print(f"keypoints_3d shape: {keypoints_3d.shape} (kpt * dims * frames)")
print(f"color shape:        {color.shape} (frames * kpt * rgb)")

In [ ]:
keypoints_3d_one_frame = keypoints_3d[:,:,0] #(kpt * dims * frames)
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(keypoints_3d_one_frame)
pcd.colors = o3d.utility.Vector3dVector(color[0,:,:]/255.0)
o3d.io.write_point_cloud("./data.ply", pcd, write_ascii=True)
o3d.visualization.draw_geometries([pcd])
